<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L03-scorecard-versus-boosting/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L03-scorecard-versus-boosting/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/credit-underwriting/lessons/P05-L03-scorecard-versus-boosting/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/credit-underwriting/lessons/P05-L03-scorecard-versus-boosting/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P05-L03 · Scorecards versus boosting: a monotone-constrained challenger and a stability report

**You will build:** a gradient-boosted challenger, written from scratch in numpy, whose trees
refuse any split that would make predicted default risk fall as an input that should raise it
rises, and a stability report that sets it beside the weight-of-evidence scorecard you built in
`P04-L07`: Gini and the population stability index, vintage by vintage, on months neither model
has seen. The last thing you write is a **test that fails a model whose predicted risk moves the
wrong way for any of the reference applications you give it**, and you will use it on both.

**Time:** ~2.5 hours · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (what a declared compute budget means),
`P04-L07-challenger-from-scratch` (a weight-of-evidence scorecard whose bins are pooled to a
documented direction, carried into this notebook as given code) and
`P05-L01-pd-vintages-and-macro` (an out-of-time split by origination month). Python and numpy;
no pandas, no scikit-learn and no LightGBM: the scorecard and the trees are in this notebook.

By the end you will be able to:

1. Implement the second-order boosting step for the logistic loss (gradient, hessian,
   regularised leaf weight, loss-reduction gain) and a histogram split search that refuses
   candidate splits which break a stated monotone direction.
2. Grow depth-limited trees that hand bounds down after every constrained split, and measure
   what a check made at the split alone lets through.
3. Implement the boosting loop and choose the number of trees on an out-of-time validation
   vintage rather than on the development data.
4. Implement a monotonicity test (every reference row, a grid taken from the model's own
   thresholds) and use it to fail an unconstrained booster, a split-only constraint and, if
   one deserves it, a scorecard.
5. Implement a Gini-and-PSI stability table by vintage with its edges cut once on development,
   and measure what the monotone constraint costs in development fit and what it buys on
   shifted out-of-time months, over replicate books from the same generator.

**The data are synthetic and the generator is in this notebook.** Nothing is downloaded. A
planted truth is what lets the notebook say that a dip in a fitted curve is *not there* in the
world, which no real book can tell you. What you conclude here about *method* transfers; what
you conclude about *these numbers* describes no real lender.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `P04-L07`, `P05-L01`

**This lesson assumes you can already:**

- write numpy code that indexes arrays with integer and boolean masks, and uses np.bincount (with weights) and np.cumsum on the result
- explain what a logistic regression predicts, and what the gradient and the hessian of a loss are (its first and second derivatives with respect to the model's output)
- read a decision tree as a set of if-then splits that put each row into one leaf, and say what the value stored in a leaf is
- read a Gini coefficient as a measure of how well scores rank outcomes, and a population stability index as a measure of how far a distribution has moved from a baseline

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Introduction to Boosted Trees (XGBoost documentation)](https://xgboost.readthedocs.io/en/stable/tutorials/model.html) · The XGBoost developers · *free* — the second-order objective with gradient and hessian statistics, the regularised leaf weight and the structure score that this lesson's leaf_value and split gain are built from
- [Monotonic Constraints (XGBoost documentation)](https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html) · The XGBoost developers · *free* — increasing and decreasing constraints written as inequalities on the model, and the note that constraints can leave histogram trees shallower
- [Parameters (LightGBM documentation)](https://lightgbm.readthedocs.io/en/stable/Parameters.html) · Microsoft and the LightGBM contributors · *free* — the monotone_constraints, monotone_constraints_method and monotone_penalty entries: the library behaviour you could check your own booster against
- [The Elements of Statistical Learning, 2nd edition](https://hastie.su.domains/ElemStatLearn/download.html) · Hastie, Tibshirani and Friedman (Springer; the authors host the free PDF) · *free* — chapter 10, Boosting and Additive Trees: boosting as stagewise fitting of trees to a gradient, the idea your fit_booster loop implements
- [Intelligent Credit Scoring, 2nd edition](https://www.wiley.com/en-us/intelligent-credit-scoring-building-and-implementing-better-credit-risk-scorecards-2nd-edition-p-9781119279150) · Naeem Siddiqi (Wiley, 2017) · *paid* — the publisher's index lists the Population Stability report (pp. 366-372), scorecard monitoring reports (pp. 360-377) and out-of-time stability validation (p. 276): the practitioner's version of this report

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
import warnings
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The book. Four inputs per application, an outcome (1 = defaults), and an origination month.
# SEED is the date this lesson was written; replicate books use SEED + 7919 k.
SEED = 20261014
N_PER_MONTH = 1200       # applications per origination month
N_MONTHS = 24
PERIODS = {"development": (0, 12), "validation": (12, 16), "out-of-time test": (16, 24)}
SHIFT_FROM = 12          # the applicant mix starts to move with this origination month
FEATURES = ("utilisation", "age of account", "delinquencies", "enquiries")
# The documented direction of RISK for each input: +1 means predicted risk must never fall as the
# input rises, -1 that it must never rise. This is knowledge about credit, written down before
# any model is fitted, exactly as in P04-L07.
DIRECTIONS = np.array([1, -1, 1, 1])
UTIL = 0                 # the column the lesson keeps coming back to

# The boosted trees.
N_BINS = 32              # histogram bins per input (the most a tree can tell apart)
LEARNING_RATE = 0.1
MAX_DEPTH = 3
LAMBDA = 1.0             # L2 regularisation on every leaf weight
MIN_CHILD_HESS = 3.0     # a child must carry at least this much hessian (about 40 applications at this book's
                         # average risk, fewer where risk is higher, because a riskier application carries more)
MAX_ROUNDS = 150         # the first cap on trees; the validation months pick fewer (the sweep prints how many)
HARD_ROUNDS = 600        # the most a fit may ever grow if the validation loss is still falling at the cap
HESS_FLOOR = 1e-12       # the logistic hessian p (1 - p) underflows to 0 at extreme scores

# The scorecard and the stability report.
WOE_BINS = 8             # quantile classes for the continuous inputs, before pooling
PSI_BINS = 10
PSI_FLOOR = 1e-6         # the floor on a bin's share, as in P04-L01
REPLICATES = 6           # books per sweep: one book is one draw
ICE_ROWS = 400           # reference rows the monotonicity test is run on


def _book_seed(k: int) -> int:
    """The seed of replicate book k (k = 0 is the book the lesson walks through)."""
    return SEED + 7919 * k


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("grad_hess",),
    "exercise 2": ("leaf_value",),
    "exercise 3": ("best_split",),
    "exercise 4": ("grow_tree",),
    "exercise 5": ("fit_booster",),
    "exercise 6": ("staged_log_loss",),
    "exercise 7": ("monotone_violations",),
    "exercise 8": ("stability_table",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_TIMED_OUT: dict[str, tuple] = {}   # label -> the functions under test when its check last ran out of time
_TIME_UP = "still running after"    # how the time limit below words itself
S: dict[str, Any] = {}         # results the demos hand to one another
_PRODUCER = {"entries": "four models", "info": "four models", "gini": "four models", "truth_gini": "four models",
             "verdicts": "the test, on every model", "sweep": "replicate books"}


def _ready(demo: str, *keys: str) -> bool:
    """True when the cells that fill `S[key]` have finished; otherwise name them and say the demo is skipped."""
    missing = sorted({_PRODUCER[k] for k in keys if k not in S})
    if not missing:
        return True
    print(f"{demo}: skipped — it needs the output of {', '.join(missing)}, which did not finish.")
    return False


# Heavy demos (fits, sweeps over many books) run when this file is the program, in a script or
# in a notebook kernel, and are skipped when a grader merely imports it to test your functions.
_RUNNING = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (calibration_bands)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 180


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: int, memory_mib: float = 2048):
    """Turn a check that runs far too long into a failure, instead of a notebook that hangs.

    Uses the operating system's alarm, so it works where that exists (Linux, macOS, Colab) and is
    skipped, harmlessly, where it does not. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, when the check ends.
    """
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"still running after {seconds} s. Look for a loop or a recursion whose stopping condition can never "
               "be met, or for a Python loop over every row where numpy would do")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _functions_under_test(label: str, needs: tuple[str, ...] = ()) -> tuple:
    """The function objects an exercise check depends on, as they stand now: a new `def` gives a new object."""
    return tuple(globals().get(name) for ex in (*needs, label) if ex in _EXERCISES for name in _EXERCISES[ex])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (), limit: int = 0,
         heavy: bool = False) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    The exercise checks run in milliseconds, so an exercise's check gets `_CHECK_SECONDS` (2 s) and
    any other cell `_DEMO_SECONDS`, unless a heavy demo asks for more with `limit`; memory that grows
    by more than 2 GiB stops it too. A loop that never ends is stopped before it can fill the
    machine's memory.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. A check that ran out of time is also recorded in `_TIMED_OUT`, together with the
    functions it was testing, so that the board does not spend the same time again on code you
    have not changed. A `heavy` demo is skipped when this file is imported rather than run.
    """
    if heavy and not _RUNNING:
        return
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    _TIMED_OUT.pop(label, None)
    try:
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS)):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        if str(exc).startswith(_TIME_UP):
            _TIMED_OUT[label] = _functions_under_test(label, needs)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def sigmoid(x: Any) -> Any:
    """Numerically safe logistic function 1 / (1 + exp(-x)). Given to you."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out[()]


def auc_gini(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Tie-aware AUC and Gini (2 * AUC - 1), higher score = more likely y = 1. Given to you.

    The rank form of the AUC from P04-L04 (carried into P04-L07): ranks with ties sharing the
    average rank, so a tied pair counts one half. Raises ValueError for one class, unequal lengths or a
    y that is not 0/1.
    """
    s = np.asarray(score, dtype=float)
    t = np.asarray(y)
    if s.ndim != 1 or t.ndim != 1 or s.size != t.size:
        raise ValueError("score and y must be 1-D arrays of equal length")
    if not np.all((t == 0) | (t == 1)):
        raise ValueError("y must contain only 0 and 1")
    n1, n0 = int(np.sum(t == 1)), int(np.sum(t == 0))
    if n1 == 0 or n0 == 0:
        raise ValueError("y must contain both classes")
    order = np.argsort(s, kind="stable")
    ordered = s[order]
    starts = np.flatnonzero(np.concatenate(([True], ordered[1:] != ordered[:-1])))
    ends = np.concatenate((starts[1:], [s.size]))
    ranks = np.empty(s.size)
    ranks[order] = np.repeat((starts + ends + 1) / 2.0, ends - starts)
    auc = (ranks[t == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc), float(2.0 * auc - 1.0)

## 1. A book in which the truth is known

Read the generator below: you are allowed to know exactly how this world works, which is the
one advantage a synthetic book has over a real one. Each application has four inputs that a
lender records at the door and an outcome, and the **planted truth** is

- **monotone in every input**: risk rises with utilisation, with delinquencies and (weakly)
  with enquiries, and falls with the age of the oldest account;
- **smooth, not stepped**: utilisation enters as a convex rising curve and the age of the
  account as a convex falling one; delinquencies enter as a capped square root and enquiries as
  a capped straight line;
- **one interaction**: a delinquency costs more on a young account than on an old one. An
  additive scorecard cannot draw that; a tree can. It is still monotone in every input, so a
  monotone constraint is correct everywhere in this world.

The applicant mix **moves from origination month 12 on**: utilisation drifts up, accounts get
younger, delinquencies and enquiries become more common. The function that turns inputs into
default risk does *not* change, so a model that learned the function and not the sample should
survive the move, and one that learned the sample's quirks may not. The months are used as in
`P05-L01`: **development** (build both models), **validation** (choose how many trees, and
nothing else) and **out-of-time test** (judge and report stability, once).

In [ ]:
class Book(NamedTuple):
    X: np.ndarray        # (n, 4) inputs: utilisation, age of account (months), delinquencies, enquiries
    y: np.ndarray        # (n,) 1 = defaulted within the outcome window, 0 = repaid
    vintage: np.ndarray  # (n,) origination month, 0 .. N_MONTHS - 1
    pd_true: np.ndarray  # (n,) the planted probability of default: for checking, never for fitting


B0 = -4.3                # intercept of the true log-odds


def true_logit(X: np.ndarray) -> np.ndarray:
    """The planted log-odds of default for each row of raw inputs. Monotone in every input."""
    u, age, d, q = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return (B0 + 2.4 * u + 1.6 * u ** 3 - 0.55 * np.log(age / 72.0)
            + 0.55 * np.sqrt(np.minimum(d, 6.0)) + 0.10 * np.minimum(q, 8.0)
            + 1.1 * np.sqrt(np.minimum(d, 4.0)) * np.exp(-age / 36.0))


def application_book(seed: int, n_per_month: int = N_PER_MONTH) -> Book:
    """Deterministic book of `N_MONTHS * n_per_month` applications. Same seed, same book."""
    rng = np.random.default_rng(seed)
    n = n_per_month * N_MONTHS
    vintage = np.repeat(np.arange(N_MONTHS), n_per_month)
    shift = np.clip((vintage - (SHIFT_FROM - 1)) / 12.0, 0.0, 1.0)   # 0 before the move, 1 at the end
    stress = rng.standard_normal(n)                                  # a latent factor the inputs share
    z_util = 0.5 * stress + np.sqrt(0.75) * rng.standard_normal(n)
    util = np.clip(0.40 + 0.20 * z_util + 0.15 * shift, 0.0, 1.2)
    age = 72.0 * np.exp(0.65 * rng.standard_normal(n) - 0.40 * shift)
    delinq = rng.poisson(0.22 * np.exp(0.5 * stress + 0.25 * shift)).astype(float)
    enq = rng.poisson(1.2 * np.exp(0.3 * stress + 0.20 * shift)).astype(float)
    X = np.column_stack([util, age, delinq, enq])
    pd_true = sigmoid(true_logit(X))
    y = (rng.random(n) < pd_true).astype(int)
    return Book(X, y, vintage, pd_true)


def period_mask(vintage: np.ndarray, name: str) -> np.ndarray:
    """Boolean mask of the rows originated in the named period."""
    lo, hi = PERIODS[name]
    return (vintage >= lo) & (vintage < hi)


BOOK = application_book(SEED)
DEV, VAL, TEST = (period_mask(BOOK.vintage, name) for name in PERIODS)
print(f"{'period':<18}{'months':>8}{'rows':>8}{'defaults':>10}{'rate':>8}"
      f"{'mean util.':>12}{'median age':>12}{'true Gini':>11}")
for _name, _mask in zip(PERIODS, (DEV, VAL, TEST)):
    _lo, _hi = PERIODS[_name]
    print(f"{_name:<18}{f'{_lo}-{_hi - 1}':>8}{int(_mask.sum()):>8,}{int(BOOK.y[_mask].sum()):>10,}"
          f"{BOOK.y[_mask].mean():>8.1%}{BOOK.X[_mask, 0].mean():>12.3f}"
          f"{np.median(BOOK.X[_mask, 1]):>12.1f}{auc_gini(BOOK.pd_true[_mask], BOOK.y[_mask])[1]:>11.3f}")
print("\n'true Gini' is the Gini of the planted probability itself on each period's outcomes: what a model")
print("that knew the function exactly would score. A model that scores above it on the rows it was fitted on is")
print("evidence of fitted noise: nothing but the sample's luck lets a ranking beat the true probability.")


def _probe_truth_monotone() -> None:
    """Probe the planted truth on a grid, row by row, so that a later 'dip' is known to be spurious."""
    n_rows, n_grid = 500, 60
    rows = BOOK.X[np.random.default_rng(1).choice(len(BOOK.X), n_rows, replace=False)]
    for j, name in enumerate(FEATURES):
        lo, hi = np.quantile(BOOK.X[:, j], (0.01, 0.99))
        grid = np.linspace(lo, hi, n_grid)
        curves = np.empty((len(rows), len(grid)))
        for k, v in enumerate(grid):
            probe = rows.copy()
            probe[:, j] = v
            curves[:, k] = true_logit(probe)
        worst = np.max(DIRECTIONS[j] * -np.diff(curves, axis=1))
        assert worst <= 1e-12, f"the planted truth falls with {name} (by {worst:.3g}): a generator bug"
    print(f"the planted truth moves the documented way in all {len(FEATURES)} inputs: probed on "
          f"{len(rows)} rows by a {n_grid}-point grid each.")


_probe_truth_monotone()

## 2. The champion, carried in from P04-L07

The champion is the scorecard you built in `P04-L07`: every input cut into classes, each class
given its **weight of evidence** (the log of the non-defaulters' share over the defaulters'
share, so a higher weight means a safer class), neighbouring classes **pooled** whenever their
default rates move against the input's documented direction, and a logistic regression on the
weights. Its code is below, **given to you**: this lesson spends its time on the boosting side
and on stability, not on binning again. Four things differ from `P04-L07` and none changes the
method: the logistic fit is a plain Newton fit, where `P04-L07` had you write the IRLS fit; the
continuous inputs are cut into eight quantile classes (`WOE_BINS`) where `P04-L07` used five; the
two count inputs are classed by an analyst's list (0, 1, 2, 3, 4+ delinquencies, for example)
because a quantile cut of a count that is mostly zero cannot make eight classes; and
`P04-L07`'s pooling refused an empty class with a clear error, where this copy would stop with a
division by zero on one, so a copy for real data should refuse it as `P04-L07` does.

Read the three things that matter for what follows. A class lookup makes the scorecard a
**step function** of each input, with its steps at the class edges. Pooling to the documented
direction makes every input's weights move the right way. And the logistic regression is the one
place a coefficient can still come out with the wrong sign, which would turn that input's
monotone steps upside down: the test you write in section 15 is what finds out.

In [ ]:
def quantile_edges(x: np.ndarray, n_bins: int) -> np.ndarray:
    """Quantile edges of `x`, de-duplicated, with the outer edges opened to infinity. Given."""
    edges = np.unique(np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def population_stability_index(expected: np.ndarray, actual: np.ndarray, edges: np.ndarray,
                               floor: float = PSI_FLOOR) -> "StabilityResult":
    """PSI of `actual` against the baseline `expected`, on `edges` cut ONCE on the baseline. Given.

    The PSI of P04-L01. Bin i is [edges[i], edges[i+1]); values outside the edges fall in the
    nearest end bin; shares are taken over each side's own total and floored at `floor` on both
    sides before the logarithm; PSI is the sum of (actual - expected) * ln(actual / expected).
    """
    edges = np.asarray(edges, dtype=float)
    if edges.size < 2 or not np.all(np.diff(edges) > 0):
        raise ValueError("edges needs at least two strictly increasing values")
    n_bins = edges.size - 1

    def _shares(x: np.ndarray) -> np.ndarray:
        v = np.asarray(x, dtype=float)
        if v.size == 0:
            raise ValueError("population_stability_index needs a non-empty sample on both sides")
        idx = np.clip(np.searchsorted(edges, v, side="right") - 1, 0, n_bins - 1)
        return np.bincount(idx, minlength=n_bins).astype(float) / v.size

    e_pct, a_pct = _shares(expected), _shares(actual)
    e_safe, a_safe = np.maximum(e_pct, floor), np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(float(contributions.sum()), contributions, e_pct, a_pct)


class StabilityResult(NamedTuple):
    """PSI and the per-bin arithmetic behind it."""
    psi: float
    contributions: np.ndarray
    expected_pct: np.ndarray
    actual_pct: np.ndarray


def woe_from_counts(events: np.ndarray, non_events: np.ndarray, smoothing: float = 0.5) -> np.ndarray:
    """Weight of evidence per bin: ln of the non-events' smoothed share over the events'. Given.

    With a = smoothing and m the number of occupied bins, an occupied bin's weight is
    ln(((non_i + a) / (N_non + a m)) / ((ev_i + a) / (N_ev + a m))); an empty bin is nan.
    """
    e, g = np.asarray(events, dtype=float), np.asarray(non_events, dtype=float)
    occ = (e + g) > 0
    m = int(occ.sum())
    woe = np.full(e.size, np.nan)
    woe[occ] = np.log(((g[occ] + smoothing) / (g.sum() + smoothing * m))
                      / ((e[occ] + smoothing) / (e.sum() + smoothing * m)))
    return woe


class WoeTable(NamedTuple):
    """One input's classes. The last entry of each array is the MISSING class (empty here)."""
    edges: np.ndarray       # value-class edges
    events: np.ndarray      # defaults per value class, then the missing class
    non_events: np.ndarray
    woe: np.ndarray


def woe_table(x: np.ndarray, y: np.ndarray, edges: np.ndarray) -> WoeTable:
    """Class `x` on `edges` (a value on an interior edge joins the upper class) and weigh each class."""
    xa, ya = np.asarray(x, dtype=float), np.asarray(y)
    n_value = edges.size - 1
    miss = np.isnan(xa)
    idx = np.clip(np.searchsorted(edges, xa[~miss], side="right") - 1, 0, n_value - 1)
    yv = ya[~miss]
    events = np.append(np.bincount(idx, weights=(yv == 1).astype(float), minlength=n_value),
                       (ya[miss] == 1).sum()).astype(np.int64)
    non_events = np.append(np.bincount(idx, weights=(yv == 0).astype(float), minlength=n_value),
                           (ya[miss] == 0).sum()).astype(np.int64)
    return WoeTable(edges, events, non_events, woe_from_counts(events, non_events))


def enforce_monotonic(table: WoeTable, direction: str) -> WoeTable:
    """Pool adjacent value classes until the default rate moves only in `direction`. Given.

    "increasing" means the default rate may never fall from one class to the next. Equal rates are
    not a violation; a merged class is compared with the one before it again (pooling adjacent
    violators); the missing class is untouched. Your exercise in P04-L07.
    """
    ev, ne, edges = table.events, table.non_events, table.edges
    n_value = edges.size - 1
    sign = 1.0 if direction == "increasing" else -1.0
    blocks: list[list[int]] = []          # [events, non-events, index of the closing edge]
    for i in range(n_value):
        blocks.append([int(ev[i]), int(ne[i]), i + 1])
        while len(blocks) >= 2:
            (e1, n1, _), (e2, n2, end) = blocks[-2], blocks[-1]
            if sign * (e2 / (e2 + n2) - e1 / (e1 + n1)) < 0.0:
                blocks[-2:] = [[e1 + e2, n1 + n2, end]]
            else:
                break
    new_edges = np.concatenate(([edges[0]], [edges[b[2]] for b in blocks]))
    events = np.array([b[0] for b in blocks] + [int(ev[-1])], dtype=np.int64)
    non_events = np.array([b[1] for b in blocks] + [int(ne[-1])], dtype=np.int64)
    return WoeTable(new_edges, events, non_events, woe_from_counts(events, non_events))


def apply_woe(x: np.ndarray, table: WoeTable) -> np.ndarray:
    """Replace each value by its class's weight of evidence."""
    xa = np.asarray(x, dtype=float)
    n_value = table.edges.size - 1
    idx = np.clip(np.searchsorted(table.edges, xa, side="right") - 1, 0, n_value - 1)
    return table.woe[:n_value][idx]


def fit_logistic(X: np.ndarray, y: np.ndarray, ridge: float = 1e-6) -> np.ndarray:
    """Logistic regression by Newton's method with a tiny ridge; intercept first. Given.

    A plain version of the IRLS fit of P04-L07: it stops when no coefficient moves by more than
    1e-9, and there is no separation in this book to guard against.
    """
    A = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    yv = np.asarray(y, dtype=float)
    coef = np.zeros(A.shape[1])
    for _ in range(50):
        p = sigmoid(A @ coef)
        grad = A.T @ (p - yv) + ridge * coef
        hess = (A * (p * (1.0 - p))[:, None]).T @ A + ridge * np.eye(A.shape[1])
        step = np.linalg.solve(hess, grad)
        coef = coef - step
        if np.abs(step).max() < 1e-9:
            break
    return coef


# The analyst's classes for the two count inputs (the continuous inputs are cut into WOE_BINS
# quantile classes on the development rows). Fixed before any model was fitted.
COUNT_CLASSES = {2: np.array([-np.inf, 1.0, 2.0, 3.0, 4.0, np.inf]),
                 3: np.array([-np.inf, 1.0, 2.0, 3.0, 4.0, 6.0, np.inf])}


class Scorecard(NamedTuple):
    tables: tuple          # one pooled WoeTable per input
    coef: np.ndarray       # intercept first, then one coefficient per input (on its weights of evidence)


def fit_scorecard(X: np.ndarray, y: np.ndarray) -> Scorecard:
    """Fit the champion: classes, weights of evidence, pooling to the documented direction, logistic."""
    tables = []
    for j in range(X.shape[1]):
        edges = COUNT_CLASSES[j] if j in COUNT_CLASSES else quantile_edges(X[:, j], WOE_BINS)
        raw = woe_table(X[:, j], y, edges)
        tables.append(enforce_monotonic(raw, "increasing" if DIRECTIONS[j] > 0 else "decreasing"))
    design = np.column_stack([apply_woe(X[:, j], tables[j]) for j in range(X.shape[1])])
    return Scorecard(tuple(tables), fit_logistic(design, y))


def scorecard_predict(card: Scorecard, X: np.ndarray) -> np.ndarray:
    """Predicted probability of default for each row of raw inputs."""
    design = np.column_stack([apply_woe(X[:, j], card.tables[j]) for j in range(X.shape[1])])
    return sigmoid(card.coef[0] + design @ card.coef[1:])


def scorecard_thresholds(card: Scorecard, j: int) -> np.ndarray:
    """The raw values at which the scorecard's score for input `j` can change: its class edges."""
    e = card.tables[j].edges
    return e[np.isfinite(e)]


def _show_champion() -> None:
    card = fit_scorecard(BOOK.X[DEV], BOOK.y[DEV])
    S["card"] = card
    print(f"{'input':<16}{'direction':>11}{'classes':>9}{'pooled to':>11}{'coefficient':>13}")
    for j, name in enumerate(FEATURES):
        raw = woe_table(BOOK.X[DEV, j], BOOK.y[DEV],
                        COUNT_CLASSES[j] if j in COUNT_CLASSES else quantile_edges(BOOK.X[DEV, j], WOE_BINS))
        print(f"{name:<16}{('rises' if DIRECTIONS[j] > 0 else 'falls'):>11}{raw.edges.size - 1:>9}"
              f"{card.tables[j].edges.size - 1:>11}{card.coef[1 + j]:>13.3f}")
    wrong = [FEATURES[j] for j in range(len(FEATURES)) if card.coef[1 + j] > 0.0]
    print("\na higher weight of evidence means a safer class, so every coefficient should be negative:")
    print("an input whose coefficient is positive has its monotone steps turned upside down.")
    print(f"coefficients with the wrong sign on this book: {', '.join(wrong) if wrong else 'none'}.")
    p = {name: scorecard_predict(card, BOOK.X[m]) for name, m in zip(PERIODS, (DEV, VAL, TEST))}
    for name, m in zip(PERIODS, (DEV, VAL, TEST)):
        print(f"scorecard Gini, {name:<17}{auc_gini(p[name], BOOK.y[m])[1]:.3f}")


_show_champion()

## 3. Gradient boosting in a page

A boosted model is a **base score** plus a sum of small regression trees, each fitted to what
the model so far gets wrong. This lesson uses the second-order form of the XGBoost paper
(`claims.yaml` records where it is stated). Write `r_i` for the model's **raw score** for
application `i` (log-odds of default), `p_i = sigmoid(r_i)`, and take the logistic loss. Then,
for every application,

- the **gradient** is `g_i = p_i - y_i`, the first derivative of the loss in `r_i`, and
- the **hessian** is `h_i = p_i (1 - p_i)`, the second.

A tree adds one constant `w` to every application that lands in the same leaf. For a leaf that
holds a set of applications, write `G` and `H` for the sums of their `g_i` and `h_i`. The
second-order approximation says the loss falls most at the weight `w* = -G / (H + lambda)`,
where `lambda` is an L2 penalty that shrinks every weight towards zero, and that a weight `w`
on a node buys a fall of `score(G, H, w) = -G w - (H + lambda) w^2 / 2`. A **split** is judged
by its gain: `score(left, w_left) + score(right, w_right) - score(parent, w_parent)`, each at its
own weight. At the optimal weights `score` is `G^2 / (2 (H + lambda))`, the per-leaf term of the
XGBoost paper's structure score (its Eq. 6) with the sign turned round, because the paper
minimises a loss and `score` is how far it falls. With open bounds the gain is then the paper's
split gain (its Eq. 7), `min_gain` playing the part of its `gamma`; a clipped weight moves it away.

Two practical points before the first exercise. The hessian `p (1 - p)` underflows to exactly 0
at extreme scores, and a weight divides by it, so this lesson **floors it at `HESS_FLOOR`**.
And a weight can be **clipped into bounds** `[lo, hi]`: in section 8 that clip is the whole
mechanism of a monotone constraint, so the leaf-weight function takes the bounds already.
The first two exercises are small and exact, and everything after them stands on them.

## 4. Exercise 1 — `grad_hess()`

Write the gradient and hessian of the logistic loss in the raw score, for a whole array at once.
The version that goes wrong is the obvious one: `1 / (1 + np.exp(-r))` overflows for a very
negative score and the notebook prints a warning, and `p * (1 - p)` is exactly 0 for a very
confident prediction, which later becomes a division by zero. Use `sigmoid`, which is given to
you above, and floor the hessian.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the derivative of `log(1 + exp(r)) - y r` with respect to `r`? Differentiate once for
the gradient and again for the hessian. Then ask what `sigmoid` returns at a score of 800, and
what `p * (1 - p)` is there.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Convert both inputs to float arrays, get the probabilities from `sigmoid` rather than writing
the formula yourself, and build the two returned arrays from them exactly as the docstring
defines them. Raise `ValueError` for arrays of different lengths or dimensions, a `y` that is
not all 0 or 1, and a NaN in `raw`.
</details>

In [ ]:
def grad_hess(raw: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """First and second derivative of the logistic loss in the raw score.

    For p = sigmoid(raw): gradient = p - y, hessian = max(p * (1 - p), HESS_FLOOR). Must not raise
    or warn at extreme scores such as +800 or -800.

    Raises ValueError if `raw` and `y` are not 1-D arrays of the same length, if `y` holds anything
    but 0 and 1, or if `raw` holds a NaN.

    Example (a score of 0 is a probability of one half):
        >>> g, h = grad_hess(np.array([0.0, 0.0]), np.array([1, 0]))
        >>> g.tolist(), h.tolist()
        ([-0.5, 0.5], [0.25, 0.25])

    Returns:
        (gradient, hessian), two float arrays of the same length as `raw`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_grad_hess() -> None:
    g, h = grad_hess(np.array([0.0, 0.0]), np.array([1, 0]))
    assert np.allclose(g, [-0.5, 0.5], rtol=0, atol=1e-15) and np.allclose(h, 0.25, rtol=0, atol=1e-15), (
        f"at a score of 0 (probability one half) got gradient {np.round(g, 4).tolist()} and "
        f"hessian {np.round(h, 4).tolist()}, expected [-0.5, 0.5] and 0.25 each: the gradient is "
        "the prediction minus the label, the hessian p (1 - p)")
    g, h = grad_hess(np.array([np.log(3.0)]), np.array([0]))
    assert abs(g[0] - 0.75) < 1e-12 and abs(h[0] - 0.1875) < 1e-12, (
        f"a score of log 3 is a probability of 0.75: expected gradient 0.75 and hessian 0.1875, "
        f"got {g[0]:.4f} and {h[0]:.4f}")
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        try:
            g, h = grad_hess(np.array([800.0, -800.0, 40.0, -40.0]), np.array([1, 1, 0, 0]))
        except Warning as exc:
            raise AssertionError(f"a score of +-800 or +-40 raised a warning ({exc}): compute the probability with "
                                 "the sigmoid that is given to you, not 1 / (1 + exp(-r))") from None
    assert np.all(np.isfinite(g)) and np.all(h == HESS_FLOOR), (
        f"at scores of +-800 and +-40 expected finite gradients and a hessian of exactly HESS_FLOOR, got "
        f"{g.tolist()} and {h.tolist()}: p (1 - p) is 0 at +800 and +40 and tiny but not 0 at -40, and below "
        "HESS_FLOOR in every case, so take the maximum with the floor rather than replacing zeros only")
    for bad in ((np.array([0.0, 1.0]), np.array([0, 1, 1])), (np.array([0.0, 1.0]), np.array([0, 2])),
                (np.array([0.0, np.nan]), np.array([0, 1])), (np.zeros((2, 2)), np.zeros((2, 2), dtype=int))):
        try:
            grad_hess(*bad)
        except ValueError:
            continue
        raise AssertionError("grad_hess must raise ValueError for arrays of different lengths, a y that is not 0 or 1, "
                             "a NaN score, or 2-D input")
    print("exercise 1 looks right — gradient p - y, hessian p (1 - p), floored, safe at extreme scores")

In [ ]:
_try("exercise 1", _check_grad_hess)

## 5. Exercise 2 — `leaf_value()`

The weight a leaf receives: `-G / (H + lambda)`, **clipped into `[lo, hi]`**. In an ordinary
boosted tree the bounds are infinite and the clip does nothing. Under a monotone constraint they
are finite, and the clip is what keeps a deep leaf on the correct side of a shallow one. Write it
once, here, because `best_split` and `grow_tree` both call it and a mistake here is a mistake
everywhere.

<details><summary>💡 Hint 1 — what to think about</summary>

The optimum of `-G w - (H + lambda) w^2 / 2` is where its derivative is zero. If the optimum
lies outside `[lo, hi]`, which weight inside the interval is best? The curve is a parabola that
opens downwards.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute the unconstrained weight, then take the larger of it and `lo` and the smaller of that
and `hi`. Return a Python float. Raise `ValueError` if `lam` is negative or NaN, if `H + lam` is
not positive (there is nothing to divide by), if `lo` is above `hi`, or if `G`, `H`, `lo` or
`hi` is NaN.
</details>

In [ ]:
def leaf_value(G: float, H: float, lam: float, lo: float = -math.inf, hi: float = math.inf) -> float:
    """The leaf weight -G / (H + lam), clipped into [lo, hi].

    Raises ValueError if lam < 0 or is NaN, if H + lam <= 0, if lo > hi, or if any argument is NaN.

    Example:
        >>> leaf_value(-4.0, 3.0, 1.0)
        1.0
        >>> leaf_value(-4.0, 3.0, 1.0, hi=0.5)
        0.5

    Returns:
        the weight as a Python float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_leaf_value() -> None:
    assert leaf_value(-4.0, 3.0, 1.0) == 1.0, (
        f"G=-4, H=3, lam=1 gave {leaf_value(-4.0, 3.0, 1.0)}, expected -G / (H + lam) = 1.0")
    assert leaf_value(-4.0, 3.0, 1.0, hi=0.5) == 0.5, (
        "an optimum of 1.0 above hi=0.5 must come back as 0.5: clip the weight into [lo, hi]")
    assert leaf_value(-4.0, 3.0, 1.0, lo=2.0, hi=3.0) == 2.0, (
        "an optimum of 1.0 below lo=2.0 must come back as 2.0: clip from below as well")
    assert leaf_value(3.0, 2.0, 1.0) == -1.0, "G=3, H=2, lam=1: the weight is -3 / 3 = -1.0"
    assert leaf_value(1.0, -0.5, 1.0) == -2.0, (
        "H = -0.5 and lam = 1: H alone is negative but H + lam = 0.5 is positive, so the weight is -1 / 0.5 = -2.0")
    assert leaf_value(3.0, 2.0, 0.0) == -1.5, "lam = 0 is a valid penalty (none): the weight is -3 / 2 = -1.5"
    assert leaf_value(-4.0, 3.0, 1.0, lo=0.3, hi=0.3) == 0.3, "lo == hi pins the weight at that value (lo above hi is the error)"
    for bad in ((1.0, 1.0, -0.5), (1.0, 0.0, 0.0), (1.0, 1.0, 1.0, 1.0, 0.0), (math.nan, 1.0, 1.0),
                (1.0, 1.0, 1.0, math.nan, 1.0), (1.0, 1.0, 1.0, -1.0, math.nan)):
        try:
            leaf_value(*bad)
        except ValueError:
            continue
        raise AssertionError(f"leaf_value{bad} must raise ValueError (a negative lam, H + lam not "
                             "positive, lo above hi, or a NaN in any argument, the bounds included)")
    print("exercise 2 looks right — the Newton weight, clipped into the bounds a node inherited")

In [ ]:
_try("exercise 2", _check_leaf_value)

## 6. Histograms, and the monotone check at a split

Trees here work on **binned** inputs. `make_edges` cuts each input once, on the development rows,
into at most `N_BINS` bins at its quantiles, and `to_bins` turns every raw value into the number
of its bin, with a value that sits exactly on an edge going to the **upper** bin, as in the
scorecard. A tree that sends "bin `<= b`" to the left is therefore asking "is the input below
`edges[b]`?", and a model's thresholds in raw units are simply the edges it splits on. Both
functions are **given to you**. Proposing candidate splits at percentiles of each input is the
idea of the approximate split-finding algorithm of the XGBoost paper (`claims.yaml`).

To find the best split of a node you do not sort anything. For each input you add up the
gradients, the hessians and the counts of the node's rows **by bin** (`np.bincount` with weights
does it in one call), and a running sum over the bins gives the left child's totals for every
threshold at once; the right child's totals are the node's totals minus those.

**The monotone check** is one more filter on the candidates. For an input that must not push
risk down (`+1`), a split is allowed only if the left child's weight is at most the right
child's, and both weights are the **clipped** weights of `leaf_value`, because those are what the
tree will really predict. A refused candidate is not a lower-ranked one: it is gone, even if it
is the best split by gain. If every candidate is refused the node becomes a leaf. The XGBoost
documentation warns of exactly this: constraints can leave a tree shallower than an unconstrained
one would be, which is a cost the lesson measures rather than asserts. The same documentation
presents constraints as a way to encode a very strong prior belief about the true relationship and
says they can then improve predictive performance; whether they do, on this book, is an empirical
question, and the notebook asks it (`claims.yaml` records both statements as read on 14 October 2026).

In [ ]:
def make_edges(X: np.ndarray, n_bins: int = N_BINS) -> list[np.ndarray]:
    """Per-input inner bin edges: quantiles of `X` (the development rows), de-duplicated. Given.

    An edge equal to an input's minimum is dropped, because it would make an empty first bin. An
    input with few distinct values (a count) gets few edges; none gets more than `n_bins - 1`.
    """
    out = []
    for j in range(X.shape[1]):
        e = np.unique(np.quantile(X[:, j], np.linspace(0.0, 1.0, n_bins + 1)[1:-1]))
        out.append(e[e > X[:, j].min()])
    return out


def to_bins(X: np.ndarray, edges: list[np.ndarray]) -> np.ndarray:
    """Integer bin of every value: bin b holds edges[b-1] <= x < edges[b]; a value ON an edge goes up. Given."""
    return np.column_stack([np.searchsorted(e, X[:, j], side="right")
                            for j, e in enumerate(edges)]).astype(np.int64)


class Split(NamedTuple):
    gain: float
    feature: int
    bin: int          # rows with Xb[:, feature] <= bin go left
    w_left: float     # the left child's weight, already clipped into [lo, hi]
    w_right: float

## 7. Exercise 3 — `best_split()`

This is the heart of the lesson: the histogram search with the gain of section 3 and the monotone
check of section 6. The docstring states every rule, because each one has a plausible wrong
version. Three of them cost real accuracy in practice: **comparing the weights before clipping**
(a child whose weight only crosses the other's because of a bound is wrongly refused or accepted),
**counting an empty child as a candidate** (a split that sends every row one way has a gain of
zero or worse and can win a comparison by a rounding error), and **treating a tie by taking the
last candidate** (a node with empty bins in between offers several thresholds that make the same
partition: take the lowest).

<details><summary>💡 Hint 1 — what to think about</summary>

Write down, for one input, the arrays you need: per-bin sums of `g`, `h` and the row count, and
their running sums over the bins. Which bin is never a useful threshold? For a threshold, which
quantities give each child's weight, and which give its hessian?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each input in order: bin the node's rows with `np.bincount`, take running sums, derive the
right child by subtraction, clip each child's weight into the bounds, mark candidates invalid
where a child is empty, light or has a non-positive `H + lam`, or where the weights break the
input's direction, compute the gain with `-inf` for the invalid ones, and take the first best.
Keep a candidate only if it beats the best of the earlier inputs strictly. Return `None` unless
the winner's gain is strictly above `min_gain`.
</details>

In [ ]:
def best_split(Xb: np.ndarray, g: np.ndarray, h: np.ndarray, rows: np.ndarray,
               constraints: np.ndarray, n_bins: int, lam: float, min_child_hess: float,
               min_gain: float, lo: float = -math.inf, hi: float = math.inf) -> Split | None:
    """Best split of the node made of `rows`, or None.

    `Xb` is the (n, d) integer array of bins in [0, n_bins); `g` and `h` the per-row gradient and
    hessian; `rows` a 1-D integer array of the node's rows; `constraints` has one entry per input:
    +1 (the prediction must not fall as the input rises), -1 (must not rise) or 0 (free); `lo` and
    `hi` are the weight bounds the node inherited.

    A CANDIDATE is an input j and a bin b in 0 .. n_bins - 2: rows with Xb[:, j] <= b go left. With
    G, H the node's sums and (GL, HL), (GR, HR) the children's:

        w_left, w_right, w_parent = leaf_value(...) of each, with lam and the bounds lo, hi
        gain = score(GL, HL, w_left) + score(GR, HR, w_right) - score(G, H, w_parent)
        score(G, H, w) = -G w - (H + lam) w^2 / 2

    It is VALID when: both children hold at least one row; HL and HR are each at least
    `min_child_hess`; HL + lam and HR + lam are positive; and, for constraints[j] = +1,
    w_left <= w_right (for -1, w_left >= w_right; equal weights are allowed).

    Return the valid candidate with the largest gain, provided the gain is STRICTLY greater than
    `min_gain`. Ties in gain go to the lowest input, then the lowest bin. Return None if no
    candidate qualifies, if the node has fewer than two rows, or if H + lam is not positive.
    Every comparison is made on the floating-point values as computed, whatever order you sum them
    in; the grader only tests ties and boundaries that are exact in binary arithmetic.

    Raises ValueError if Xb is not a 2-D integer array with values in [0, n_bins), if g, h or
    constraints do not match its shape, if a constraint is not -1, 0 or 1, if rows is not a 1-D
    integer array of valid row numbers, if n_bins < 2, if lam or min_child_hess is negative, or
    if lo > hi.

    Example (eight rows, one input in four bins; the first four rows push risk UP, the last four DOWN):
        >>> Xb = np.array([[0], [0], [1], [1], [2], [2], [3], [3]])
        >>> g = np.array([-1.0, -1, -1, -1, 1, 1, 1, 1]); h = np.ones(8)
        >>> best_split(Xb, g, h, np.arange(8), np.array([0]), 4, 0.0, 1.0, 0.0)
        Split(gain=4.0, feature=0, bin=1, w_left=1.0, w_right=-1.0)
        >>> best_split(Xb, g, h, np.arange(8), np.array([1]), 4, 0.0, 1.0, 0.0) is None
        True

    Returns:
        a Split, or None.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_best_split() -> None:
    Xb = np.array([[0], [0], [1], [1], [2], [2], [3], [3]])
    g = np.array([-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0])
    h = np.ones(8)
    rows = np.arange(8)
    s = best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 1.0, 0.0)
    assert s is not None and (s.feature, s.bin) == (0, 1) and abs(s.gain - 4.0) < 1e-12 \
        and abs(s.w_left - 1.0) < 1e-12 and abs(s.w_right + 1.0) < 1e-12, (
        f"got {s}, expected gain 4.0 at bin 1 with weights +1 and -1: the first four rows have gradient -1 "
        "(weight +1), the last four +1 (weight -1); the score of a child is -G w - (H + lam) w^2 / 2")
    assert best_split(Xb, g, h, rows, np.array([1]), 4, 0.0, 1.0, 0.0) is None, (
        "with constraint +1 the only sensible split puts the HIGHER weight on the left, so it must be "
        "refused, and no other split exists to take its place: expected None")
    s = best_split(Xb, g, h, rows, np.array([-1]), 4, 0.0, 1.0, 0.0)
    assert s is not None and s.bin == 1, "with constraint -1 (risk must not rise) the same split is allowed"
    s = best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 1.0, 0.0, lo=-0.5, hi=0.5)
    assert s is not None and abs(s.w_left - 0.5) < 1e-12 and abs(s.w_right + 0.5) < 1e-12 \
        and abs(s.gain - 3.0) < 1e-12, (
        f"with bounds [-0.5, 0.5] expected weights 0.5 and -0.5 and gain 3.0, got {s}: clip each child's "
        "weight into the bounds BEFORE scoring it")
    Xp, gp = np.array([[0]] * 4 + [[1]] * 4), np.array([-1.0] * 4 + [0.5] * 4)
    s = best_split(Xp, gp, h, rows, np.array([0]), 2, 1.0, 1.0, 0.0)
    assert s is not None and abs(s.gain - 16.0 / 9.0) < 1e-12 and abs(s.w_left - 0.8) < 1e-12 \
        and abs(s.w_right + 0.4) < 1e-12, (
        f"four rows with gradient -1 and four with +0.5, hessian 1 each, lam = 1: the children weigh 0.8 and -0.4 and score "
        f"1.6 and 0.4, the parent (G = -2, H = 8) weighs 2/9 and scores 2/9, so the gain is 1.6 + 0.4 - 2/9 = 1.7778; got "
        f"{s}. Every score uses H + lam, and the parent's own score is subtracted")
    s = best_split(Xp, gp, h, rows, np.array([0]), 2, 1.0, 1.0, -1.0, lo=-0.1, hi=0.1)
    assert s is not None and abs(s.gain - 0.395) < 1e-12 and abs(s.w_left - 0.1) < 1e-12 and abs(s.w_right + 0.1) < 1e-12, (
        f"the same rows with bounds [-0.1, 0.1]: the children clip to 0.1 and -0.1 (scores 0.375 and 0.175) and so does the "
        f"parent (2/9 becomes 0.1, score 0.155), so the gain is 0.395; got {s}. The parent is clipped like every other node")
    pair, gpair = np.array([[0], [0], [1], [1]]), np.array([-1.0, -1.0, 1.0, 1.0])
    s = best_split(pair, gpair, np.ones(4), np.arange(4), np.array([1]), 2, 0.0, 1.0, -1.0, lo=0.3, hi=0.3)
    assert s is not None and abs(s.w_left - 0.3) < 1e-12 and abs(s.w_right - 0.3) < 1e-12, (
        f"bounds [0.3, 0.3] pin both children at 0.3, and equal weights are allowed, although the unclipped weights (+1 "
        f"and -1) are out of order for a +1 input; got {s}. Compare the CLIPPED weights, and let lo == hi through")
    assert best_split(np.full((5, 2), 2), np.array([-1.0, 1.0, -1.0, 1.0, -1.0]), np.ones(5), np.arange(5),
                      np.array([0, 0]), 4, 1.0, 0.0, -1.0) is None, (
        "every row sits in the same bin, so every candidate leaves a child with no rows: expected None, although an "
        "empty child's gain of 0 is above min_gain = -1")
    assert best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 4.5, 0.0) is None, (
        "min_child_hess = 4.5 leaves no candidate whose children both carry that much hessian (the "
        "best has 4.0 each): expected None")
    assert best_split(Xb, g, h, rows, np.array([0]), 4, 1.0, 4.5, 0.0) is None, (
        "the same node with lam = 1.0: min_child_hess = 4.5 is compared with each child's hessian sum (4.0 at best), "
        "not with H + lam (5.0), so expected None. lam belongs in the weight and the score, not in this rule")
    assert best_split(Xb, g, 0.25 * h, rows, np.array([0]), 4, 0.0, 2.0, 0.0) is None, (
        "every row has a hessian of 0.25, so the best split's children carry 1.0 each: min_child_hess = 2.0 asks "
        "for more hessian than they have, although each child holds four rows. The rule is about the SUM of the "
        "hessians, not the number of rows: expected None")
    s = best_split(Xb, g, 2.0 * h, rows, np.array([0]), 4, 0.0, 5.0, 0.0)
    assert s is not None and s.bin == 1, (
        "every row has a hessian of 2.0, so each four-row child carries 8.0, which is enough for min_child_hess = 5.0 "
        f"although it holds fewer than five rows; got {s}")
    s = best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 4.0, 0.0)
    assert s is not None and s.bin == 1, (
        f"each child of the best split carries a hessian of exactly 4.0, and min_child_hess = 4.0 asks for AT LEAST 4.0 "
        f"on BOTH sides; got {s}")
    assert best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 4.0000001, 0.0) is None, (
        "min_child_hess a hair above 4.0 asks for more than either child carries: expected None")
    assert best_split(Xb, g, h, rows, np.array([0]), 4, 0.0, 1.0, 4.0) is None, (
        "the best gain is exactly 4.0 and min_gain is 4.0: the gain must be STRICTLY greater, so None")
    twin = np.column_stack([Xb[:, 0], Xb[:, 0]])
    s = best_split(twin, g, h, rows, np.array([0, 0]), 4, 0.0, 1.0, 0.0)
    assert s is not None and s.feature == 0, (
        f"two identical inputs tie exactly; the lowest input must win, got input {None if s is None else s.feature}")
    s = best_split(twin, g, h, rows, np.array([1, 0]), 4, 0.0, 1.0, 0.0)
    assert s is not None and s.feature == 1, (
        f"input 0 must not push risk down and this split puts the higher weight on its left, so input 0 is refused; input 1 "
        f"is free and offers the same split: expected input 1, got {None if s is None else s.feature}. Each input has its own "
        "constraint")
    s = best_split(Xb, g, h, np.arange(6), np.array([0]), 4, 0.0, 1.0, 0.0)
    assert s is not None and s.bin == 1 and abs(s.gain - 8.0 / 3.0) < 1e-12, (
        f"only the first six rows belong to this node (bins 0, 0, 1, 1, 2, 2): the best split is still after bin 1, but its "
        f"gain is 2 + 1 - 1/3 = 8/3, not the 4.0 of all eight rows; got {s}. Use only the rows in `rows`")
    zero_h = np.array([1.0, 1.0, 0.0, 0.0])
    assert best_split(np.array([[0], [0], [1], [1]]), np.array([-1.0, -1.0, 1.0, 1.0]), zero_h, np.arange(4),
                      np.array([0]), 2, 0.0, 0.0, 0.0) is None, (
        "the right child has a hessian of 0 and lam = 0, so H + lam is 0 and there is nothing to divide by: that candidate is "
        "invalid, and it is the only one, so expected None")
    assert best_split(np.array([[0], [0], [1], [1]]), np.array([-1.0, -1.0, 1.0, 1.0]), zero_h, np.arange(4),
                      np.array([0]), 2, 0.0, 0.0, 0.0, -0.5, 0.5) is None, (
        "the same zero-hessian child under finite bounds: -G / 0 clips to a bound and looks like a weight, but H + lam is still "
        "0, so the candidate is still invalid: expected None")
    assert best_split(np.array([[0], [1], [2], [3], [0]]), np.array([-1.0, 1.0, -1.0, 1.0, -1.0]), np.zeros(5),
                      np.arange(5), np.array([0]), 4, 0.0, 0.0, -1.0) is None, (
        "a node whose hessians sum to 0 with lam = 0 has H + lam = 0: return None, there is no weight to compute")
    sparse = np.array([[0], [0], [3], [3]])
    s = best_split(sparse, np.array([-1.0, -1.0, 1.0, 1.0]), np.ones(4), np.arange(4), np.array([0]), 4,
                   0.0, 1.0, 0.0)
    assert s is not None and s.bin == 0, (
        f"bins 0, 1 and 2 all make the same partition when bins 1 and 2 are empty; got bin "
        f"{None if s is None else s.bin}, expected the LOWEST, 0")
    for bad in (dict(constraints=np.array([2])), dict(n_bins=1), dict(lo=1.0, hi=0.0), dict(lam=-1.0)):
        kw = dict(Xb=Xb, g=g, h=h, rows=rows, constraints=np.array([0]), n_bins=4, lam=0.0,
                  min_child_hess=1.0, min_gain=0.0)
        kw.update(bad)
        try:
            best_split(**kw)
        except ValueError:
            continue
        raise AssertionError(f"best_split must raise ValueError for {list(bad)}")
    try:
        best_split(np.zeros((8, 1), dtype=int), g, h, rows, np.array([0]), 1, 0.0, 1.0, 0.0)
    except ValueError:
        pass
    else:
        raise AssertionError("n_bins = 1 must raise ValueError even when every row sits in bin 0: a candidate needs "
                             "at least two bins")
    print("exercise 3 looks right — histogram gain, clipped weights, the monotone check, strict gain, stated ties")

In [ ]:
_try("exercise 3", _check_best_split)

## 8. A tree, and why checking the split is not enough

A `Tree` is five parallel arrays (the input split on at each node, the bin, the left and right
child, and the weight of a leaf), and `predict_tree` walks each row down it. Both are **given to
you**. The node numbering is yours to choose: nothing reads it except `predict_tree`.

The check in `best_split` compares the two children of **one** split. A tree has splits below
them. Picture a split on utilisation that sends the low-utilisation applications left at weight
0.0 and the high ones right at weight 0.5: allowed. Now the left child splits again, on another
input, and one of its new leaves takes the weight 1.0. That leaf is on the low-utilisation side
of the first split yet predicts more than everything on the high side, so raise utilisation for
an applicant who lands in it and the prediction falls. No single split broke the rule. The tree
did.

**The cure is to hand bounds down.** After a split on a constrained input, take the midpoint of
the two children's weights. For a `+1` input the left subtree may never predict above it and the
right subtree never below it (a `-1` input mirrors that). Each child inherits its new bound
together with the ones its parent had, and every leaf beneath is clipped into them, which is the
`lo` and `hi` that `leaf_value` and `best_split` already take. A split on an unconstrained input
hands both children the parent's bounds unchanged. LightGBM's default constraint method and its
split-time rejection of out-of-order children do the same (`claims.yaml` gives the source as read
on 14 October 2026), and its documentation says this method over-constrains: a leaf is held at
the midpoint though it could have gone further. That cost shows up in the monotone booster's
development Gini, mixed with the cost of the constraint itself; the lesson does not separate the two.

In [ ]:
class Tree(NamedTuple):
    feature: np.ndarray   # int per node: the input split on, or -1 at a leaf
    bin: np.ndarray       # int per node: rows with Xb[:, feature] <= bin go left
    left: np.ndarray      # int per node: index of the left child, -1 at a leaf
    right: np.ndarray     # int per node: index of the right child, -1 at a leaf
    value: np.ndarray     # float per node: the leaf weight (0.0 at an internal node)


def make_tree(nodes: list) -> Tree:
    """Build a Tree from a list of (feature, bin, left, right, value) tuples; node 0 is the root. Given."""
    cols = list(zip(*nodes))
    return Tree(np.array(cols[0], dtype=int), np.array(cols[1], dtype=int),
                np.array(cols[2], dtype=int), np.array(cols[3], dtype=int),
                np.array(cols[4], dtype=float))


def predict_tree(tree: Tree, Xb: np.ndarray) -> np.ndarray:
    """The weight of the leaf each row of `Xb` lands in. Given."""
    node = np.zeros(len(Xb), dtype=int)
    while True:
        f = tree.feature[node]
        live = np.flatnonzero(f >= 0)
        if live.size == 0:
            return tree.value[node]
        go_left = Xb[live, f[live]] <= tree.bin[node[live]]
        node[live] = np.where(go_left, tree.left[node[live]], tree.right[node[live]])

## 9. Exercise 4 — `grow_tree()`

Grow one tree, depth-first, from all the rows. A node at depth `d` may be split only while
`d < max_depth`; it becomes a leaf when `best_split` returns `None`, with the weight of
`leaf_value` under the bounds it inherited. After a split on a constrained input and with
`propagate_bounds=True`, compute the midpoint of the **clipped** child weights the split returned
and pass the bounds down as described. With `propagate_bounds=False` the tree is the
**ablation** the next sections test: every child gets the open interval `(-inf, +inf)`, so only
the split itself is checked. A mistake that passes a plain unconstrained test is passing the
parent's bounds to both children after a constrained split.

<details><summary>💡 Hint 1 — what to think about</summary>

Take a `+1` input and a split with weights 0.0 and 0.5. Which interval should the left subtree
live in, and which the right? If the left child splits again and its right grandchild wants 0.9,
what should the tree predict there, and what does the bound do to the gain `best_split` computes
for that second split?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Write a small recursive function of (the node's rows, its depth, its bounds) that appends a node
to a list, asks `best_split` unless the depth limit is reached, and either finishes a leaf or
splits the rows with `Xb[rows, feature] <= bin`, builds the two children with their bounds and
records their indices. Finish with `make_tree`. Raise `ValueError` for a negative `max_depth` or
for no rows.
</details>

In [ ]:
def grow_tree(Xb: np.ndarray, g: np.ndarray, h: np.ndarray, constraints: np.ndarray, max_depth: int,
              n_bins: int, lam: float, min_child_hess: float, min_gain: float,
              propagate_bounds: bool = True) -> Tree:
    """Grow one depth-limited tree on all the rows of `Xb`.

    The root holds every row, at depth 0, with bounds (-inf, +inf). A node at depth d is split
    only while d < max_depth, by `best_split` under the node's bounds; if that returns None the
    node is a leaf of weight `leaf_value(G, H, lam, lo, hi)`. After a split on an input with a
    non-zero constraint, with mid = (w_left + w_right) / 2:

        propagate_bounds=True,  constraint +1: left child (lo, mid), right child (mid, hi)
                                constraint -1: left child (mid, hi), right child (lo, mid)
        propagate_bounds=False: both children (-inf, +inf)

    After a split on a free input (0) both children inherit (lo, hi). Node numbering is free.

    Raises ValueError if `max_depth` is negative or `Xb` has no rows.

    Example (sixteen rows; input 0 must not push risk down; the left half of input 0 holds a big
    positive leaf under input 1 that the bounds must pull down to the midpoint 0.25):
        >>> x0 = np.repeat([0, 1, 2, 3], 4); x1 = np.tile([0, 0, 1, 1], 4)
        >>> Xb = np.column_stack([x0, x1])
        >>> g = np.where(x0 <= 1, np.where(x1 == 0, 1.0, -1.0), np.where(x1 == 0, -1.5, 0.5))
        >>> probe = np.array([[1, 0], [1, 1], [2, 0], [2, 1]])
        >>> t = grow_tree(Xb, g, np.ones(16), np.array([1, 0]), 2, 4, 0.0, 1.0, 0.0)
        >>> predict_tree(t, probe).tolist()
        [-1.0, 0.25, 1.5, 0.25]
        >>> t = grow_tree(Xb, g, np.ones(16), np.array([1, 0]), 2, 4, 0.0, 1.0, 0.0,
        ...               propagate_bounds=False)
        >>> predict_tree(t, probe).tolist()
        [-1.0, 1.0, 1.5, -0.5]

    Returns:
        a Tree.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _fixture_two_inputs() -> tuple:
    x0 = np.repeat([0, 1, 2, 3], 4)
    x1 = np.tile([0, 0, 1, 1], 4)
    g = np.where(x0 <= 1, np.where(x1 == 0, 1.0, -1.0), np.where(x1 == 0, -1.5, 0.5))
    return np.column_stack([x0, x1]), g, np.ones(16)


def _check_grow_tree() -> None:
    Xb, g, h = _fixture_two_inputs()
    probe = np.array([[1, 0], [1, 1], [2, 0], [2, 1]])
    cons = np.array([1, 0])
    t = grow_tree(Xb, g, h, cons, 2, 4, 0.0, 1.0, 0.0)
    got = predict_tree(t, probe).tolist()
    assert got == [-1.0, 0.25, 1.5, 0.25], (
        f"with inherited bounds expected [-1.0, 0.25, 1.5, 0.25], got {got}. The root splits input 0 with "
        "weights 0.0 and 0.5, so the midpoint is 0.25: the left subtree may not predict above it (its "
        "weight of 1.0 becomes 0.25) and the right subtree may not predict below it (-0.5 becomes 0.25)")
    t = grow_tree(Xb, g, h, cons, 2, 4, 0.0, 1.0, 0.0, propagate_bounds=False)
    got = predict_tree(t, probe).tolist()
    assert got == [-1.0, 1.0, 1.5, -0.5], (
        f"with propagate_bounds=False expected [-1.0, 1.0, 1.5, -0.5], got {got}: only the split is checked, "
        "so each child subtree is free, and input 1 at its high bin FALLS from 1.0 to -0.5 as input 0 rises")
    t = grow_tree(Xb, -g, h, np.array([-1, 0]), 2, 4, 0.0, 1.0, 0.0)
    got = predict_tree(t, probe).tolist()
    assert got == [1.0, -0.25, -1.5, -0.25], (
        f"every gradient negated and input 0 constrained -1 (risk must not rise): the tree is the mirror image of the "
        f"first one, every weight negated, so expected [1.0, -0.25, -1.5, -0.25], got {got}. The midpoint is -0.25, and "
        "for a -1 input the LEFT subtree may not go below it and the RIGHT subtree may not go above it")
    stump = grow_tree(Xb, g, h, cons, 0, 4, 0.0, 1.0, 0.0)
    assert len(stump.value) == 1 and stump.feature[0] == -1, "max_depth=0 must give a single leaf"
    assert len(grow_tree(Xb, g, h, cons, 3, 4, 0.0, 1.0, 1e9).value) == 1, (
        "a min_gain that no split can beat must leave a single leaf: pass min_gain on to best_split")
    assert abs(stump.value[0] - leaf_value(float(g.sum()), 16.0, 0.0)) < 1e-12, (
        "a tree of depth 0 is one leaf with the weight leaf_value(G, H, lam) of all the rows")
    # every input-0 curve of the bounded tree must be monotone, at every setting of input 1
    full = np.array([[a, b] for a in range(4) for b in range(2)])
    pred = predict_tree(grow_tree(Xb, g, h, cons, 3, 4, 0.0, 1.0, 0.0), full).reshape(4, 2)
    assert np.all(np.diff(pred, axis=0) >= 0), (
        f"the bounded tree falls somewhere as input 0 rises: predictions by input 0 (rows) and input 1 "
        f"(columns) are {pred.tolist()}")
    # a second grid, in which a bound CLIPS a child's weight: both inputs must not push risk down
    grid = np.array([[a, b] for a in range(4) for b in range(4)])
    g2 = np.array([0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 1.0, 0.0, -2.0, 0.0])
    got = predict_tree(grow_tree(grid, g2, np.ones(16), np.array([1, 1]), 3, 4, 0.0, 1.0, 0.0), grid)
    want = np.array([-2 / 3] * 3 + [-1 / 6] + [-1 / 6] + [1 / 6] * 3 + [-1 / 6] + [1 / 6] * 3 + [-1 / 6] + [2 / 3] * 3)
    assert np.allclose(got, want, rtol=0, atol=1e-12), (
        f"a 4 x 4 grid of rows (input 0 down, input 1 across), both inputs +1: expected {np.round(want, 4).tolist()}, got "
        f"{np.round(got, 4).tolist()}. The root splits input 0 with weights -1/2 and 1/6, so the midpoint is -1/6. In the "
        "right subtree the best split is on input 1, and its left child's own weight, -1/3, is below that bound, so "
        "best_split returns it clipped to -1/6: the midpoint handed down is made from the CLIPPED weights -1/6 and 1/3 "
        "(that is 1/12), not from -1/3 and 1/3")
    try:
        grow_tree(Xb, g, h, cons, -1, 4, 0.0, 1.0, 0.0)
    except ValueError:
        pass
    else:
        raise AssertionError("a negative max_depth must raise ValueError")
    print("exercise 4 looks right — bounds go down the tree, and the ablation shows what they prevent")

In [ ]:
_try("exercise 4", _check_grow_tree, needs=("exercise 2", "exercise 3"))

## 10. Exercise 5 — `fit_booster()`

The boosting loop. Start from the **base score**, the log-odds of the training default rate, so
that the first tree has something sensible to correct. Then, for each round: take the gradient
and hessian at the model's current raw scores (`grad_hess`), grow a tree on them (`grow_tree`),
and move every raw score by the tree's weight **times the learning rate**. The learning rate is
the one number that has to be applied in two places, here and when predicting, and a booster
that applies it in only one of them fits one model and predicts another.

<details><summary>💡 Hint 1 — what to think about</summary>

After round `t` the model's raw score for a row is the base score plus the learning rate times
the sum of the first `t` trees' weights for that row. What does the next round need to know about
the model so far, and in what form do you already hold it? What is the base score if half the
rows default? If a quarter do?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check the arguments, compute the base score, keep an array of raw scores that starts at the base
for every row, and loop `n_rounds` times: gradient and hessian, a tree, an update of the raw
scores by `learning_rate * predict_tree(...)`, the tree appended to a list. Return the base, the
learning rate and the trees. Raise `ValueError` for a `y` that is not 0/1 or has one class, a
negative `n_rounds` or a `learning_rate` that is not positive.
</details>

In [ ]:
class Booster(NamedTuple):
    base: float            # the raw score every row starts from
    learning_rate: float
    trees: list            # the trees, in the order they were grown


def predict_raw(booster: Booster, Xb: np.ndarray, n_trees: int | None = None) -> np.ndarray:
    """Raw score (log-odds) of each row using the first `n_trees` trees (all by default). Given."""
    k = len(booster.trees) if n_trees is None else n_trees
    out = np.full(len(Xb), booster.base)
    for tree in booster.trees[:k]:
        out = out + booster.learning_rate * predict_tree(tree, Xb)
    return out


def fit_booster(Xb: np.ndarray, y: np.ndarray, constraints: np.ndarray, n_rounds: int,
                learning_rate: float = LEARNING_RATE, max_depth: int = MAX_DEPTH, n_bins: int = N_BINS,
                lam: float = LAMBDA, min_child_hess: float = MIN_CHILD_HESS, min_gain: float = 0.0,
                propagate_bounds: bool = True) -> Booster:
    """Fit a gradient-boosted classifier with the logistic loss.

    base = ln(p / (1 - p)) with p the mean of `y`; raw starts at `base` for every row; each of
    `n_rounds` rounds takes `grad_hess(raw, y)`, grows `grow_tree(...)` on them and adds
    `learning_rate * predict_tree(tree, Xb)` to `raw`.

    Raises ValueError if `y` is not 1-D of 0 and 1 with both classes present and one entry per row
    of `Xb`, if `n_rounds` is negative, or if `learning_rate` is not positive.

    Example (a lone input that separates the classes; the first tree already splits it):
        >>> Xb = np.array([[0]] * 4 + [[1]] * 4); y = np.array([0, 0, 0, 1, 0, 1, 1, 1])
        >>> b = fit_booster(Xb, y, np.array([0]), 1, 1.0, 1, 2, 0.0, 0.5, 0.0)
        >>> predict_raw(b, np.array([[0], [1]])).tolist()
        [-1.0, 1.0]

    Returns:
        a Booster.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit_booster() -> None:
    Xb = np.array([[0]] * 4 + [[1]] * 4)
    y = np.array([0, 0, 0, 1, 0, 1, 1, 1])
    b = fit_booster(Xb, y, np.array([0]), 1, 1.0, 1, 2, 0.0, 0.5, 0.0)
    got = predict_raw(b, np.array([[0], [1]])).tolist()
    assert np.allclose(got, [-1.0, 1.0], rtol=0, atol=1e-12), (
        f"one round, learning rate 1, expected raw scores [-1.0, 1.0], got {got}: the base score is 0 (half the "
        "rows default), the gradients are 0.5 or -0.5, the hessians 0.25, so each leaf has G = +-1, H = 1")
    b = fit_booster(Xb, y, np.array([0]), 1, 0.5, 1, 2, 0.0, 0.5, 0.0)
    got = predict_raw(b, np.array([[0], [1]])).tolist()
    assert np.allclose(got, [-0.5, 0.5], rtol=0, atol=1e-12), (
        f"learning rate 0.5 must halve the step: expected [-0.5, 0.5], got {got}")
    skew = fit_booster(Xb, np.array([0, 0, 0, 0, 0, 0, 1, 0]), np.array([0]), 0, 0.1, 1, 2, 0.0, 0.5, 0.0)
    assert len(skew.trees) == 0 and abs(skew.base - math.log(1.0 / 7.0)) < 1e-12, (
        f"with no rounds the model is its base score, ln(p / (1 - p)) = ln(1/7) for one default in eight; "
        f"got base {skew.base:.4f} and {len(skew.trees)} trees")
    lopsided = fit_booster(Xb, np.array([0, 0, 0, 0, 0, 0, 1, 1]), np.array([0]), 1, 1.0, 1, 2, 0.0, 0.5, 0.0)
    want = [math.log(1.0 / 3.0) - 4.0 / 3.0, math.log(1.0 / 3.0) + 4.0 / 3.0]
    got = predict_raw(lopsided, np.array([[0], [1]])).tolist()
    assert np.allclose(got, want, rtol=0, atol=1e-12), (
        f"two defaults in eight, both in the second group, one round at learning rate 1: expected "
        f"{np.round(want, 4).tolist()}, got {np.round(got, 4).tolist()}. The base score is ln(1/3), so each gradient is "
        "0.25 - y and each hessian 0.1875, and the first round must start from that base. A round that starts from "
        "a raw score of 0 gets the same answer only when exactly half of the rows default")
    two = fit_booster(Xb, y, np.array([0]), 2, 0.5, 1, 2, 0.0, 0.5, 0.0)
    assert len(two.trees) == 2 and not np.array_equal(two.trees[1].value, two.trees[0].value), (
        "the second tree must be fitted to gradients taken at the model AFTER the first (the raw scores must "
        "be updated every round), so it cannot be a copy of the first")

    def newton(ys: list, rounds: int, lr: float, raw: float) -> float:
        for _ in range(rounds):                      # one group of rows that share a leaf, lam = 0
            p = 1.0 / (1.0 + math.exp(-raw))
            raw += lr * (-sum(p - t for t in ys) / (len(ys) * p * (1.0 - p)))
        return raw

    two = fit_booster(Xb, np.array([0, 0, 0, 0, 0, 0, 1, 1]), np.array([0]), 2, 0.5, 1, 2, 0.0, 0.1, 0.0)
    want = [newton([0, 0, 0, 0], 2, 0.5, math.log(1.0 / 3.0)), newton([0, 0, 1, 1], 2, 0.5, math.log(1.0 / 3.0))]
    got = predict_raw(two, np.array([[0], [1]])).tolist()
    assert np.allclose(got, want, rtol=0, atol=1e-12), (
        f"two defaults in eight, two rounds at learning rate 0.5: expected {np.round(want, 4).tolist()}, got "
        f"{np.round(got, 4).tolist()}. Each group's Newton step is -G / H at ITS OWN current score; the second round "
        "starts from the scores after the first round's SHRUNKEN step (learning rate times weight), and the booster "
        "must predict with the same learning rate it trained with")
    refused = fit_booster(Xb, np.array([0, 0, 0, 0, 0, 0, 1, 1]), np.array([-1]), 2, 0.5, 1, 2, 0.0, 0.1, 0.0)
    got = predict_raw(refused, np.array([[0], [1]])).tolist()
    assert np.allclose(got, [math.log(1.0 / 3.0)] * 2, rtol=0, atol=1e-12), (
        f"the only split puts the HIGHER-risk group on the right, which a -1 constraint (risk must not rise) refuses, so every "
        f"tree is a single leaf of weight 0 and the model stays at its base score ln(1/3) = {math.log(1 / 3):.4f}; got "
        f"{np.round(got, 4).tolist()}. Pass the constraints on to grow_tree")
    cells = np.array([[a, b] for a in range(4) for b in range(2)])
    Xg = np.array([[a, b] for a in range(4) for b in range(2) for _ in range(4)])
    defaults = [[0, 3], [0, 3], [4, 2], [4, 2]]       # defaults among the four rows of each (input 0, input 1) cell
    yg = np.array([int(k < defaults[a][b]) for a in range(4) for b in range(2) for k in range(4)])
    steps = {}
    for propagate in (True, False):
        booster = fit_booster(Xg, yg, np.array([1, 0]), 1, 1.0, 2, 4, 1.0, 0.5, 0.0, propagate)
        steps[propagate] = np.diff(predict_raw(booster, cells).reshape(4, 2), axis=0).min()
    assert steps[True] >= -1e-12, (
        f"with propagate_bounds=True the booster FALLS by {-steps[True]:.4f} as input 0 rises: hand the bounds down in every round")
    assert steps[False] < -0.1, (
        f"propagate_bounds=False must give the split-only booster, which falls on these 32 rows (its worst step is "
        f"{steps[False]:.4f}): pass propagate_bounds on to grow_tree. If this is 0 the flag changes nothing")
    for bad in (dict(y=np.zeros(8, dtype=int)), dict(y=np.ones(8, dtype=int)), dict(n_rounds=-1), dict(learning_rate=0.0)):
        kw = dict(Xb=Xb, y=y, constraints=np.array([0]), n_rounds=1, learning_rate=0.5, max_depth=1, n_bins=2,
                  lam=0.0, min_child_hess=0.5, min_gain=0.0)
        kw.update(bad)
        try:
            fit_booster(**kw)
        except ValueError:
            continue
        raise AssertionError(f"fit_booster must raise ValueError for {list(bad)} (a y of one class only has no base score: "
                             "check it before taking the logarithm)")
    print("exercise 5 looks right — base score, then gradient, tree and shrunken update, round by round")

In [ ]:
_try("exercise 5", _check_fit_booster, needs=("exercise 1", "exercise 4"))

## 11. Exercise 6 — `staged_log_loss()`

How many trees? Too few underfit and too many fit the sample's noise. The usual answer is to
watch the loss on data the model was not fitted on and stop where it is lowest. This lesson uses
the **validation months** for that: later than development, so the answer is found on a
population that has already begun to move, the way the model will be used. It is a design
choice, not a measured advantage, and it has a price: those months are never fitted on.

Write the function that makes that curve: the mean logistic loss after **zero** trees (the base
score alone), after one, after two, and so on, and the number of trees where it is lowest. On a
tie take the **fewest** trees. It must not refit anything and must not walk the whole ensemble
once per number of trees: add one tree at a time.

<details><summary>💡 Hint 1 — what to think about</summary>

The loss for a row with raw score `r` and label `y` is `log(1 + exp(r)) - y r`. For which `r` does
`exp(r)` overflow, and what does `np.logaddexp(0, r)` do instead? After `k` trees, what must you
add to the raw scores to get to `k + 1`?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start the raw scores at the base, record the loss, then for each tree add its shrunken weights
and record again, so a booster with `T` trees gives `T + 1` numbers. `np.argmin` returns the
first minimum. Raise `ValueError` for a `y` that is not 0/1 or whose length differs from the
rows of `Xb`.
</details>

In [ ]:
def staged_log_loss(booster: Booster, Xb: np.ndarray, y: np.ndarray) -> tuple[int, np.ndarray]:
    """Mean log loss after 0, 1, ..., T trees, and the number of trees where it is lowest.

    losses[k] is the mean over rows of log(1 + exp(r)) - y r, with r the raw score from the base
    and the first k trees (shrunken by the learning rate). `best` is the smallest k attaining the
    minimum, so a tie goes to the fewer trees (two losses are tied when they are equal as floating-
    point numbers). Must not overflow at extreme scores.

    Raises ValueError if `y` is not 1-D of 0 and 1 with one entry per row of `Xb`.

    Example (a base of 0 and one stump that is right about both rows):
        >>> stump = make_tree([(0, 0, 1, 2, 0.0), (-1, -1, -1, -1, -1.0), (-1, -1, -1, -1, 1.0)])
        >>> best, losses = staged_log_loss(Booster(0.0, 1.0, [stump]), np.array([[0], [1]]), np.array([0, 1]))
        >>> best, [round(float(v), 4) for v in losses]
        (1, [0.6931, 0.3133])

    Returns:
        (best, losses): an int and a float array of length len(booster.trees) + 1.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_staged_log_loss() -> None:
    stump = make_tree([(0, 0, 1, 2, 0.0), (-1, -1, -1, -1, -1.0), (-1, -1, -1, -1, 1.0)])
    flipped = make_tree([(0, 0, 1, 2, 0.0), (-1, -1, -1, -1, 1.0), (-1, -1, -1, -1, -1.0)])
    Xb, y = np.array([[0], [1]]), np.array([0, 1])
    best, losses = staged_log_loss(Booster(0.0, 1.0, [stump, flipped]), Xb, y)
    want = [math.log(2.0), math.log1p(math.exp(-1.0)), math.log(2.0)]
    assert np.allclose(losses, want, rtol=0, atol=1e-12) and len(losses) == 3, (
        f"expected the losses {np.round(want, 4).tolist()} (the base alone, then the right stump, then the "
        f"flipped one that undoes it), got {np.round(losses, 4).tolist()}")
    assert best == 1 and isinstance(best, (int, np.integer)), (
        f"the lowest loss is after one tree, and `best` is a whole number; got {best!r}")
    def loss(raw: list) -> float:
        return sum(math.log1p(math.exp(-abs(r))) + max(r, 0.0) - t * r for r, t in zip(raw, y)) / len(y)

    strong = make_tree([(0, 0, 1, 2, 0.0), (-1, -1, -1, -1, -2.0), (-1, -1, -1, -1, 2.0)])
    best, losses = staged_log_loss(Booster(0.5, 0.5, [strong]), Xb, y)
    want = [loss([0.5, 0.5]), loss([-0.5, 1.5])]
    assert np.allclose(losses, want, rtol=0, atol=1e-12) and best == 1, (
        f"a base score of 0.5, a learning rate of 0.5 and a stump of -2 and +2: the raw scores are 0.5, 0.5 and then "
        f"-0.5, 1.5, so expected the losses {np.round(want, 4).tolist()} and best = 1, got {np.round(losses, 4).tolist()} "
        f"and {best}. Start from the base score, and shrink every tree by the learning rate")
    best, losses = staged_log_loss(Booster(0.0, 1.0, [flipped, strong]), Xb, y)
    assert best == 2 and losses[1] > losses[0] > losses[2], (
        f"a tree that makes things worse, then one that more than undoes it: the loss goes up and then down to its lowest "
        f"value, after both trees, so best = 2; got {best} from the losses {np.round(losses, 4).tolist()}. Take the "
        "position of the overall minimum, not the first place the loss stops falling")
    zero = make_tree([(-1, -1, -1, -1, 0.0)])
    best, losses = staged_log_loss(Booster(0.0, 1.0, [zero, zero]), Xb, y)
    assert best == 0 and np.allclose(losses, math.log(2.0)), (
        f"three equal losses must report the FEWEST trees (0), got {best}")
    huge = make_tree([(-1, -1, -1, -1, 900.0)])
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        try:
            _, big = staged_log_loss(Booster(0.0, 1.0, [huge]), Xb, y)
        except Warning as exc:
            raise AssertionError(f"a raw score of 900 raised a warning ({exc}): use np.logaddexp(0, r), not "
                                 "np.log(1 + np.exp(r))") from None
    assert np.isfinite(big).all(), f"a raw score of 900 gave the non-finite losses {big.tolist()}"
    print("exercise 6 looks right — the loss after every number of trees, and the fewest trees at the minimum")

In [ ]:
_try("exercise 6", _check_staged_log_loss)

In [ ]:
class Challenger(NamedTuple):
    """A fitted booster with what it needs to score raw inputs. Given."""
    booster: Booster          # truncated to the number of trees the validation months chose
    edges: list               # the bin edges the trees were grown on (development rows only)
    constraints: np.ndarray


def fit_challenger(book: Book, constraints: np.ndarray, propagate_bounds: bool = True,
                   max_rounds: int = MAX_ROUNDS) -> tuple[Challenger, np.ndarray]:
    """Fit on the development rows, then keep the number of trees that is best on the validation
    months. Returns the model and the validation loss by number of trees. Given.

    If the validation loss is still lowest at the last tree, the cap was too low for this book: the
    model is refitted with twice the rounds, up to `HARD_ROUNDS`, so the answer never depends on an
    arbitrary cap. (The first fit's trees are the first trees of the second, so nothing is lost.)
    """
    dev, val = period_mask(book.vintage, "development"), period_mask(book.vintage, "validation")
    edges = make_edges(book.X[dev])
    Xb_dev, Xb_val = to_bins(book.X[dev], edges), to_bins(book.X[val], edges)
    rounds = max_rounds
    while True:
        booster = fit_booster(Xb_dev, book.y[dev], constraints, rounds, propagate_bounds=propagate_bounds)
        best, losses = staged_log_loss(booster, Xb_val, book.y[val])
        if best < rounds or rounds >= HARD_ROUNDS:
            break
        rounds = min(2 * rounds, HARD_ROUNDS)
    return Challenger(booster._replace(trees=booster.trees[:best]), edges, constraints), losses


def challenger_predict(model: Challenger, X: np.ndarray) -> np.ndarray:
    """Predicted probability of default for each row of raw inputs. Given."""
    return sigmoid(predict_raw(model.booster, to_bins(X, model.edges)))


def challenger_thresholds(model: Challenger, j: int) -> np.ndarray:
    """The raw values input `j` is split at anywhere in the ensemble: where its curve can change. Given."""
    found = [model.edges[j][t.bin[n]] for t in model.booster.trees for n in range(len(t.feature))
             if t.feature[n] == j]
    return np.unique(np.array(found, dtype=float))


def mean_leaves(model: Challenger) -> float:
    """The average number of leaves per tree. Given."""
    return float(np.mean([int((t.feature < 0).sum()) for t in model.booster.trees])) if model.booster.trees else 0.0

## 12. Four models on the same months

Time to put the pieces to work. `fit_challenger` (given, above) fits a booster on the development
months and keeps the number of trees your `staged_log_loss` finds best on the validation months.
The cell fits **three boosters** that differ only in how they are constrained, and sets them
beside the champion:

- **unconstrained boosting**: every input free, as a data scientist would run it by default;
- **monotone boosting**: the documented direction on every input, with bounds handed down;
- **split-only constraint**: the same directions, but the ablation of exercise 4: each split is
  checked and no bounds are inherited.

Every model is judged on the same three sets of months. The planted truth is in the first row,
so the table shows what a model that knew the function exactly would score on these outcomes. A
second table sets each model's average predicted default probability beside the actual default
rate, because a model can rank well and still be wrong about the level of risk.

In [ ]:
def _entry(predict: Callable, thresholds: Callable) -> dict:
    return {"predict": predict, "thresholds": thresholds}


def _gini_on(predict: Callable, book: Book, mask: np.ndarray) -> float:
    return auc_gini(predict(book.X[mask]), book.y[mask])[1]


def _month_vintages(predict: Callable, book: Book) -> tuple:
    """Development scores, and the (label, score, y) of every out-of-time month. Given."""
    dev = period_mask(book.vintage, "development")
    months = [(f"month {m}", book.vintage == m) for m in range(*PERIODS["out-of-time test"])]
    return predict(book.X[dev]), [(label, predict(book.X[mk]), book.y[mk]) for label, mk in months]


def fit_four(book: Book) -> tuple[dict, dict]:
    """Fit the champion and the three boosters on one book. Returns {name: entry} and {name: info}. Given."""
    dev = period_mask(book.vintage, "development")
    card = fit_scorecard(book.X[dev], book.y[dev])
    entries = {"scorecard": _entry(lambda X: scorecard_predict(card, X), lambda j: scorecard_thresholds(card, j))}
    info = {"scorecard": {"trees": None, "leaves": None, "coef": card.coef}}
    for name, cons, prop in (("unconstrained boosting", np.zeros(len(FEATURES), dtype=int), True),
                             ("monotone boosting", DIRECTIONS, True),
                             ("split-only constraint", DIRECTIONS, False)):
        model, losses = fit_challenger(book, cons, prop)
        entries[name] = _entry(lambda X, m=model: challenger_predict(m, X),
                               lambda j, m=model: challenger_thresholds(m, j))
        info[name] = {"trees": len(model.booster.trees), "leaves": mean_leaves(model), "losses": losses}
    return entries, info


def _fit_models() -> None:
    entries, info = fit_four(BOOK)
    S["entries"], S["info"] = entries, info
    masks = (DEV, VAL, TEST)
    print(f"{'model':<24}{'trees':>7}{'leaves/tree':>13}" + "".join(f"{n.split()[0]:>14}" for n in PERIODS)
          + "    (Gini by period)")
    truth = [auc_gini(BOOK.pd_true[m], BOOK.y[m])[1] for m in masks]
    print(f"{'planted truth':<24}{'-':>7}{'-':>13}" + "".join(f"{v:>14.3f}" for v in truth))
    gini = {}
    for name, e in entries.items():
        gini[name] = [_gini_on(e["predict"], BOOK, m) for m in masks]
        trees = "-" if info[name]["trees"] is None else str(info[name]["trees"])
        leaves = "-" if info[name]["leaves"] is None else f"{info[name]['leaves']:.2f}"
        print(f"{name:<24}{trees:>7}{leaves:>13}" + "".join(f"{v:>14.3f}" for v in gini[name]))
    S["gini"], S["truth_gini"] = gini, truth
    free, mono = gini["unconstrained boosting"], gini["monotone boosting"]
    n_free, n_mono = info["unconstrained boosting"]["trees"], info["monotone boosting"]["trees"]
    sizes = (f"They stop at different numbers of trees ({n_free} and {n_mono}), each chosen on the validation months, "
             "so the development figure mixes the cost of the constraint with where each model stopped."
             if n_free != n_mono else f"Both stop at {n_free} trees, chosen on the validation months.")
    print(f"\nwhat the monotone constraint cost in development fit: {free[0] - mono[0]:+.4f} Gini "
          f"(unconstrained minus monotone); what it did out of time: {mono[2] - free[2]:+.4f} "
          f"(monotone minus unconstrained). {sizes}")
    for name in ("unconstrained boosting", "monotone boosting"):
        if gini[name][0] > truth[0]:
            print(f"{name} ranks the development rows better than the planted probability itself "
                  f"({gini[name][0]:.3f} against {truth[0]:.3f}): a model that had learned only the function "
                  "would have no systematic way to do that, so the surplus is evidence of the sample's noise, "
                  "fitted (section 17 shows how far a Gini moves from book to book).")
    print(f"mean leaves per tree: unconstrained {info['unconstrained boosting']['leaves']:.2f}, monotone "
          f"{info['monotone boosting']['leaves']:.2f} (a tree of depth {MAX_DEPTH} has at most "
          f"{2 ** MAX_DEPTH} leaves).")
    print("\naverage predicted default probability against the actual default rate, by period (a model whose")
    print("average sits far from the actual rate is mis-calibrated, whatever its Gini)")
    print(f"{'':<24}" + "".join(f"{n.split()[0]:>14}" for n in PERIODS))
    print(f"{'actual default rate':<24}" + "".join(f"{BOOK.y[m].mean():>14.2%}" for m in masks))
    for name, e in entries.items():
        print(f"{name:<24}" + "".join(f"{e['predict'](BOOK.X[m]).mean():>14.2%}" for m in masks))


_try("four models", _fit_models, needs=tuple(f"exercise {i}" for i in range(1, 7)), limit=150, heavy=True)

## 13. What each model does with utilisation

Gini says how well a model ranks; it does not say whether the curve it draws makes sense. The
next cell asks each model, for one input, a plain question: *take real applications from the
out-of-time months, change only their utilisation to each value on a grid, and average the
predicted default probability*. That average is the **partial dependence** curve. The planted
truth gets the same treatment, so its curve is the one every fitted curve is trying to follow, and
it only ever rises.

In [ ]:
def _average_curve(predict: Callable, X_ref: np.ndarray, j: int, grid: np.ndarray) -> np.ndarray:
    """Mean predicted probability with input `j` set to each grid value. Given."""
    out = np.empty(len(grid))
    for k, v in enumerate(grid):
        probe = X_ref.copy()
        probe[:, j] = v
        out[k] = predict(probe).mean()
    return out


def _show_curves() -> None:
    if not _ready("utilisation curves", "entries"):
        return
    X_ref = BOOK.X[TEST][np.random.default_rng(3).choice(int(TEST.sum()), 1500, replace=False)]
    grid = np.quantile(BOOK.X[TEST, UTIL], np.linspace(0.02, 0.98, 13))
    curves = {"planted truth": _average_curve(lambda X: sigmoid(true_logit(X)), X_ref, UTIL, grid)}
    for name, e in S["entries"].items():
        curves[name] = _average_curve(e["predict"], X_ref, UTIL, grid)
    print("average predicted default rate as utilisation is set to each value (out-of-time rows)\n")
    names = list(curves)
    print(f"{'utilisation':>12}" + "".join(f"{n.split()[0] if n != 'planted truth' else 'truth':>16}" for n in names))
    for k, v in enumerate(grid):
        print(f"{v:>12.3f}" + "".join(f"{curves[n][k]:>16.2%}" for n in names))
    top = float(S["entries"]["scorecard"]["thresholds"](UTIL).max())
    print(f"\nthe scorecard's top utilisation class starts at {top:.3f}: {np.mean(BOOK.X[DEV, UTIL] >= top):.1%} of "
          f"development applications are in it and {np.mean(BOOK.X[TEST, UTIL] >= top):.1%} of out-of-time ones, and "
          "inside a class the scorecard cannot tell one applicant from another.")
    falls = {n: int(np.sum(np.diff(c) < 0)) for n, c in curves.items()}
    print("\nsteps along the grid at which the AVERAGE curve falls: "
          + ", ".join(f"{n} {f}" for n, f in falls.items()) + f" (of {len(grid) - 1}).")
    print("An average can hide a dip that individual applications show, and can show one that only some")
    print("applications have; the test of section 14 looks at every application separately.")
    S["curves"] = (grid, curves)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    style = {"planted truth": ("black", "-", 2.4), "scorecard": ("tab:blue", "-", 1.6),
             "unconstrained boosting": ("tab:red", "-", 1.6), "monotone boosting": ("tab:green", "-", 1.6),
             "split-only constraint": ("tab:orange", "--", 1.4)}
    for n, c in curves.items():
        colour, ls, lw = style[n]
        ax.plot(grid, c, color=colour, ls=ls, lw=lw, label=n)
    ax.set_xlabel("utilisation set to this value")
    ax.set_ylabel("average predicted default probability")
    ax.set_title("What each model does with utilisation (out-of-time rows)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_try("utilisation curves", _show_curves, needs=tuple(f"exercise {i}" for i in range(1, 7)), limit=150, heavy=True)

## 14. The test that fails a model

The requirement is one sentence: *no model goes forward if its predicted default risk ever falls
as utilisation rises* (and the same for every input with a documented direction). Written as a
test it can only examine the applications it is given, so it is written for a set of reference
rows, and it has to be exact enough to find what the models can hide. Three decisions make it so.

1. **Every reference row, separately.** An average of curves can rise even though individual
   curves fall (two applications that dip in different places average out), and can fall though
   most individual curves do not. The test varies one input for each reference row on its own.
2. **A grid that cannot step over a dip.** A tree ensemble and a scorecard are both *step
   functions* of an input: they change value only at their own thresholds, and a value exactly on
   a threshold belongs to the bin above it. So a grid with one point below the lowest threshold
   and one **at each threshold** has a point in every step the model has, so along the input being
   varied the test is exhaustive rather than sampled (across the other inputs it covers only the
   reference rows it is given). `exact_grid` (given) builds it. An equally spaced grid can land
   both its neighbours either side of a one-bin dip.
3. **A tolerance that is stated in arithmetic.** The fall between two neighbouring grid points
   is `before - after`, the prediction at the earlier point minus the prediction at the later
   one (for a decreasing input the sign is turned round), computed in floating point. A fall is
   counted when it is greater than `tol`, so a model flat between two points passes, and with
   `tol = 0.0` any fall at all fails. Nothing here is rounded.

Exercise 7 writes the function that does the looking; `assert_risk_monotone`, given below, turns
its answer into the test.

In [ ]:
class Violations(NamedTuple):
    n_rows: int           # reference rows whose curve falls (in the wrong direction) by more than tol
    n_tested: int         # reference rows tested
    worst_drop: float     # the largest fall found, whatever tol is; 0.0 if no step falls
    worst_row: int        # the (first) row of that fall in X_ref, or -1 if no step falls
    worst_pair: tuple | None   # (grid[i], grid[i + 1]) of that fall, or None


def exact_grid(thresholds: np.ndarray, pad: float = 1.0) -> np.ndarray:
    """One point below the lowest threshold and one AT each distinct threshold, ascending. Given.

    A model that changes value only at `thresholds` (a value on a threshold goes to the upper
    bin) takes every value it can take at these points. With no thresholds the model is constant in
    this input and any two points will do.
    """
    t = np.unique(np.asarray(thresholds, dtype=float))
    if t.size == 0:
        return np.array([0.0, 1.0])
    return np.concatenate(([t[0] - pad], t))

## 15. Exercise 7 — `monotone_violations()`

Given a prediction function, a set of reference rows, an input and its documented direction,
find every reference row whose predicted risk moves the wrong way somewhere along a grid. For
every row, copy it once per grid value with that one input replaced, predict all the copies, and
compare **neighbouring** grid points. "Increasing" means risk must not fall; "decreasing" that it
must not rise. The reference rows themselves must come back untouched. A function that compares
only the first grid point with the last, or only the average over rows, passes the easy cases and
fails the ones this test exists for.

You have seen the core of it already: `_probe_truth_monotone` in section 1 varies one input over
a grid for each row of a sample of the planted truth and looks at the differences between
neighbouring grid points, times the direction. Read it first. Yours does the same for any
`predict` function, honours a tolerance, and reports how many rows fall and where the worst fall is.

<details><summary>💡 Hint 1 — what to think about</summary>

Suppose the grid has `G` points and there are `m` reference rows. How many predictions do you need
and in what shape do you want them? Which pairs of columns do you compare? For a "decreasing"
input, how does the sign of the difference you are testing change?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build an array that repeats each reference row `G` times (copy, never edit the caller's array),
put the grid into the chosen column, predict once, and reshape to `m` rows by `G` columns. The
fall between neighbours is the earlier prediction minus the later, times `-1` for a decreasing
input. Count rows with any fall above `tol`, and find the single largest fall with its row and its
pair of grid values. Raise `ValueError` for malformed arguments and for a NaN prediction: a NaN
compares as neither higher nor lower and would silently pass.
</details>

In [ ]:
def monotone_violations(predict: Callable[[np.ndarray], np.ndarray], X_ref: np.ndarray, feature: int,
                        direction: str, grid: np.ndarray, tol: float = 0.0) -> Violations:
    """Find the reference rows whose predicted risk moves the wrong way along `grid`.

    `predict` maps an (n, d) array to n predicted probabilities. For each row of `X_ref`, set column
    `feature` to each value of `grid` (all other columns kept) and predict. With p[k] the prediction
    at grid[k], the FALL at step k is p[k] - p[k + 1] for direction "increasing" and p[k + 1] - p[k]
    for "decreasing", computed in floating point. A row VIOLATES if any fall is strictly greater
    than `tol`. `worst_drop` is the largest fall over all rows and steps (0.0 if every fall is <= 0);
    `worst_row` and `worst_pair` locate it, the first row and then the first step on a tie, and are
    -1 and None when `worst_drop` is 0.0. `X_ref` is read as floats (an integer array is converted
    first, so a grid value is never truncated) and is not modified.

    Raises ValueError if X_ref is not a non-empty 2-D array, if `feature` is not a column of it, if
    `direction` is not "increasing" or "decreasing", if `grid` is not 1-D, finite, strictly
    increasing and of at least two values, if `tol` is negative or NaN, or if `predict` returns the
    wrong number of values or any NaN.

    Example (a model that drops by 2 once the input passes 1, but only for rows whose second column
    is positive):
        >>> predict = lambda X: X[:, 0] - 2.0 * ((X[:, 0] > 1.0) & (X[:, 1] > 0))
        >>> v = monotone_violations(predict, np.array([[0.0, -1.0], [0.0, 1.0]]), 0, "increasing",
        ...                         np.array([0.0, 0.5, 1.0, 1.5, 2.0]))
        >>> v
        Violations(n_rows=1, n_tested=2, worst_drop=1.5, worst_row=1, worst_pair=(1.0, 1.5))

    Returns:
        a Violations.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_monotone_violations() -> None:
    predict = lambda X: X[:, 0] - 2.0 * ((X[:, 0] > 1.0) & (X[:, 1] > 0))
    ref = np.array([[0.0, -1.0], [0.0, 1.0]])
    grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
    v = monotone_violations(predict, ref, 0, "increasing", grid)
    assert (v.n_rows, v.n_tested, v.worst_row, v.worst_pair) == (1, 2, 1, (1.0, 1.5)) and abs(v.worst_drop - 1.5) < 1e-12, (
        f"got {v}, expected one violating row (row 1) with a worst drop of 1.5 between 1.0 and 1.5: the second "
        "row's prediction is 1.0 at 1.0 and -0.5 at 1.5")
    assert ref.tolist() == [[0.0, -1.0], [0.0, 1.0]], "X_ref was modified: copy the rows before setting the input"
    assert isinstance(v.n_rows, (int, np.integer)) and isinstance(v.worst_drop, float), (
        f"n_rows must be a whole number and worst_drop a float, got {type(v.n_rows).__name__} and {type(v.worst_drop).__name__}")
    v = monotone_violations(lambda X: X[:, 0], ref, 0, "increasing", grid)
    assert v.n_rows == 0 and v.worst_drop == 0.0 and v.worst_row == -1 and v.worst_pair is None, (
        f"a model that rises everywhere must give no violation, worst_drop 0.0, worst_row -1 and worst_pair None, "
        f"got {v}")
    v = monotone_violations(lambda X: -X[:, 0], ref, 0, "decreasing", grid)
    assert v.n_rows == 0, f"-x is monotone DECREASING: expected no violation for direction 'decreasing', got {v}"
    v = monotone_violations(lambda X: X[:, 0], ref, 0, "decreasing", grid)
    assert v.n_rows == 2, f"x rises, so for direction 'decreasing' both rows violate, got {v.n_rows}"
    # curves that fall in the middle and recover: only a step-by-step comparison sees them
    dip = lambda X: np.where(np.isclose(X[:, 0], 1.0), -1.0, X[:, 0])
    v = monotone_violations(dip, ref, 0, "increasing", grid)
    assert v.n_rows == 2 and abs(v.worst_drop - 1.5) < 1e-12, (
        f"a dip at 1.0 that recovers by 1.5 is a violation (a first-versus-last comparison cannot see it); got {v}")
    v = monotone_violations(lambda X: np.where(X[:, 0] > 1.2, 0.75, X[:, 0]), ref, 0, "increasing", grid, tol=0.25)
    assert v.n_rows == 0 and abs(v.worst_drop - 0.25) < 1e-12 and v.worst_row == 0 and v.worst_pair == (1.0, 1.5), (
        f"a fall of exactly 0.25 with tol = 0.25 is allowed (a fall counts when it is GREATER than tol), "
        f"and worst_drop, worst_row and worst_pair report the largest fall anyway, whatever tol is: got {v}")
    v = monotone_violations(lambda X: np.where(X[:, 0] > 1.2, 1.0 - 1e-13, np.minimum(X[:, 0], 1.0)), ref, 0,
                            "increasing", grid)
    assert v.n_rows == 2 and 0.0 < v.worst_drop < 1e-12, (
        f"a fall of 1e-13 is still a fall: with the default tol = 0.0 any fall at all fails, however small; got {v}")
    late = lambda X: np.where(X[:, 0] >= 2.0, -5.0, X[:, 0])
    v = monotone_violations(late, ref, 0, "increasing", grid)
    assert v.n_rows == 2 and v.worst_pair == (1.5, 2.0) and abs(v.worst_drop - 6.5) < 1e-12, (
        f"a fall only at the LAST step (between 1.5 and 2.0) is still a fall: every neighbouring pair is compared, the "
        f"final one included; got {v}")
    v = monotone_violations(lambda X: -X[:, 1], ref, 1, "increasing", np.array([0.0, 1.0, 2.0]))
    assert v.n_rows == 2 and abs(v.worst_drop - 1.0) < 1e-12 and v.worst_pair == (0.0, 1.0), (
        f"column 1 is the input to vary here, and this model falls by 1.0 a step as it rises: expected both rows to violate, "
        f"got {v}. The grid goes into column `feature`, not always column 0")
    two = np.array([0.0, 1.0])
    assert monotone_violations(lambda X: X[:, 0], ref, 0, "increasing", two).n_rows == 0 and (
        monotone_violations(lambda X: -X[:, 0], ref, 0, "increasing", two).n_rows == 2), (
        "a grid of two points is valid: it has one step, and that step counts")
    shifted = np.array([[0.7, 5.0], [1.3, -5.0]])
    kept = shifted.copy()
    monotone_violations(lambda X: X[:, 0], shifted, 0, "increasing", grid)
    assert np.array_equal(shifted, kept), (
        "X_ref was modified: its first column held 0.7 and 1.3, and the grid values were written into it. Copy the rows "
        "before setting the input")
    v = monotone_violations(lambda X: np.where(np.isclose(X[:, 0], 0.5), -1.0, X[:, 0]), np.array([[0, 1], [1, 2]]), 0,
                            "increasing", np.array([0.0, 0.5, 1.0]))
    assert v.n_rows == 2 and abs(v.worst_drop - 1.0) < 1e-12, (
        f"X_ref here is an INTEGER array; the model dips at 0.5, and a copy that keeps the integer type turns the 0.5 into 0 "
        f"and never sees the dip. Convert to float before writing the grid into it; got {v}")
    for bad in (dict(grid=np.array([0.0, 1.0, 1.0])), dict(grid=np.array([1.0])), dict(direction="up"),
                dict(feature=5), dict(tol=-1.0), dict(X_ref=np.empty((0, 2))), dict(grid=np.array([0.0, np.inf])),
                dict(grid=np.array([0.0, np.nan, 2.0])), dict(grid=np.array([1.0, 0.0]))):
        kw = dict(predict=lambda X: X[:, 0], X_ref=ref, feature=0, direction="increasing", grid=grid, tol=0.0)
        kw.update(bad)
        try:
            monotone_violations(**kw)
        except ValueError:
            continue
        raise AssertionError(f"monotone_violations must raise ValueError for {list(bad)}")
    try:
        monotone_violations(lambda X: np.full(len(X), np.nan), ref, 0, "increasing", grid)
    except ValueError:
        pass
    else:
        raise AssertionError("a NaN prediction must raise ValueError: NaN compares False with everything and would pass")
    print("exercise 7 looks right — every row on its own, neighbouring grid points, a stated tolerance")

In [ ]:
_try("exercise 7", _check_monotone_violations)

In [ ]:
def assert_risk_monotone(label: str, predict: Callable, thresholds_for: Callable, X_ref: np.ndarray,
                         features: Sequence[int] | None = None, tol: float = 0.0) -> dict:
    """THE TEST: fail if the model's predicted risk moves against a documented direction for any of the
    reference rows it is given. Given.

    For each input, run `monotone_violations` over `exact_grid` of the model's own thresholds. Raise
    AssertionError naming every input that fails, with how many reference rows and the worst fall;
    otherwise return the Violations by input.
    """
    found, failing = {}, []
    for j in (range(len(FEATURES)) if features is None else features):
        v = monotone_violations(predict, X_ref, j, "increasing" if DIRECTIONS[j] > 0 else "decreasing",
                                exact_grid(thresholds_for(j)), tol)
        found[j] = v
        if v.n_rows:
            a, b = v.worst_pair
            failing.append(f"{FEATURES[j]}: {v.n_rows} of {v.n_tested} reference rows, worst fall "
                           f"{v.worst_drop:.4f} (row {v.worst_row}, between {a:.4g} and {b:.4g})")
    if failing:
        raise AssertionError(f"{label}: predicted default risk moves the wrong way for " + "; ".join(failing))
    return found


def _reference_rows() -> np.ndarray:
    """The rows the test is run on: out-of-time applications, the population the model will score."""
    pool = np.flatnonzero(TEST)
    return BOOK.X[np.sort(np.random.default_rng(5).choice(pool, ICE_ROWS, replace=False))]


def _show_test() -> None:
    if not _ready("the test, on every model", "entries"):
        return
    X_ref = _reference_rows()
    names = list(S["entries"])
    print(f"the test, run on {len(X_ref)} out-of-time reference rows with each model's own thresholds:\n")
    print(f"{'model':<24}" + "".join(f"{f:>16}" for f in FEATURES) + "   verdict")
    S["verdicts"] = {}
    for name in names:
        e = S["entries"][name]
        counts, verdict = [], "PASS"
        try:
            found = assert_risk_monotone(name, e["predict"], e["thresholds"], X_ref)
        except AssertionError as exc:
            verdict = "FAIL"
            S.setdefault("first_failure", str(exc))
            found = {j: monotone_violations(e["predict"], X_ref, j, "increasing" if DIRECTIONS[j] > 0 else "decreasing",
                                            exact_grid(e["thresholds"](j))) for j in range(len(FEATURES))}
        counts = [found[j].n_rows for j in range(len(FEATURES))]
        S["verdicts"][name] = (verdict, counts)
        print(f"{name:<24}" + "".join(f"{c:>16}" for c in counts) + f"   {verdict}")
    print("\n(each cell: reference rows whose predicted risk falls somewhere along the input's own grid)")
    print(f"models that FAIL the test: {', '.join(n for n in names if S['verdicts'][n][0] == 'FAIL') or 'none'}.")
    if "first_failure" in S:
        print("\nwhat the test says when it fails a model (the first failure above):\n  " + S["first_failure"])


_try("the test, on every model", _show_test, needs=tuple(f"exercise {i}" for i in range(1, 8)),
     limit=150, heavy=True)

## 16. Gini says whether the ranking survived, PSI says whether the population moved

They answer different questions, and a stability report needs both. The **Gini** of a vintage is
computed from its own outcomes: does the model still put the defaulters above the repayers? The
**population stability index** needs no outcome at all: it compares the distribution of the
model's scores in a vintage with their distribution on the development rows, over bins that were
cut **once, on the development scores**, and floors every share before taking the log (the PSI given
in section 2 is `P04-L01`'s). Freezing the edges is part of the definition: the development
scores are the baseline, so every month is measured against the same yardstick and the months can
be laid side by side. Bins re-cut on each month's own scores would be a different yardstick every
month; the cell after exercise 8 prints both for the first and the last test month, and the
largest gap between them over all the test months.

Exercise 8 builds the table that goes in the report: one row per vintage, in the order given, with
its size, its default rate, its Gini and its PSI. A vintage with only one class of outcome (a
month with no defaults, or nothing but defaults) has no Gini: the row reports `nan` and the table
carries on, because a stability report that stops at its first awkward month is not a report.

<details><summary>💡 Hint 1 — what to think about</summary>

Where do the bin edges come from, and how many times are they computed? What must each vintage's
score be compared with? Which exception does `auc_gini` raise for a single class, and which other
problems raise the same exception type?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Cut the edges once from the development scores with `quantile_edges`. For each vintage check that
its score and outcome arrays have the same non-zero length (raise `ValueError` if not), compute
the default rate, take the Gini only when both outcomes occur (otherwise `nan`), and the PSI of
its scores against the development scores on those edges. Raise `ValueError` too for an empty
list of vintages, for no development scores or for `n_bins` below 2.
</details>

In [ ]:
class VintageRow(NamedTuple):
    label: str
    n: int
    default_rate: float
    gini: float       # nan if the vintage has only one class of outcome
    psi: float


def stability_table(dev_score: np.ndarray, vintages: Sequence[tuple], n_bins: int = PSI_BINS) -> list:
    """One VintageRow per vintage: size, default rate, Gini and the PSI of its scores against development.

    `vintages` is a sequence of (label, score, y). The PSI edges are `quantile_edges(dev_score,
    n_bins)`, cut ONCE from the development scores and used for every vintage; the PSI is
    `population_stability_index(dev_score, score, edges).psi`. The Gini is `auc_gini(score, y)[1]`,
    or nan when `y` holds only one class. Rows come back in the order given.

    Raises ValueError if `vintages` is empty, `dev_score` is empty, `n_bins` < 2, or a vintage's
    `score` and `y` are empty or have different lengths.

    Example (a vintage that is the development sample itself has a PSI of exactly 0):
        >>> dev = np.arange(100.0)
        >>> rows = stability_table(dev, [("same", dev, np.arange(100) % 2)])
        >>> rows[0].n, rows[0].psi
        (100, 0.0)

    Returns:
        a list of VintageRow.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_stability_table() -> None:
    dev = np.arange(1000.0) / 1000.0
    y = np.arange(1000) % 2
    moved = np.clip(dev + 0.3, 0.0, 1.0)
    rows = stability_table(dev, [("same", dev, y), ("moved", moved, y), ("one class", dev, np.zeros(1000, dtype=int)),
                                 ("all defaults", dev, np.ones(1000, dtype=int))])
    assert [r.label for r in rows] == ["same", "moved", "one class", "all defaults"] and rows[0].n == 1000, (
        f"expected one row per vintage in the order given, got {[r.label for r in rows]}")
    assert rows[0].psi == 0.0, (
        f"the development sample against itself must have a PSI of exactly 0, got {rows[0].psi}")
    want = population_stability_index(dev, moved, quantile_edges(dev, PSI_BINS)).psi
    assert abs(rows[1].psi - want) < 1e-12 and rows[1].psi > 0.1, (
        f"a population shifted by 0.3: expected exactly the PSI on edges cut once from the development scores "
        f"({want:.4f}), got {rows[1].psi:.4f}. Cut the edges ONCE from the development scores and reuse them for "
        "every vintage: edges cut from the vintage itself are a different baseline for each vintage, and "
        "that is a different quantity, not a smaller one")
    assert math.isnan(rows[2].gini) and abs(rows[2].default_rate) == 0.0 and not math.isnan(rows[2].psi), (
        "a vintage with no defaults has a default rate of 0.0, a Gini of nan, and still a PSI")
    assert rows[2].psi == 0.0, (
        f"the third vintage holds the development scores themselves, so its PSI against development is exactly 0, got "
        f"{rows[2].psi:.4f}: is each vintage compared with the vintage before it? Every vintage is compared with the "
        "development scores")
    assert math.isnan(rows[3].gini) and rows[3].default_rate == 1.0 and not math.isnan(rows[3].psi), (
        "a vintage with nothing but defaults has a default rate of 1.0, a Gini of nan, and still a PSI: either single class "
        "has no Gini")
    assert abs(rows[0].gini - auc_gini(dev, y)[1]) < 1e-12, "the Gini is auc_gini(score, y)[1]"
    five = stability_table(dev, [(7, moved, y)], n_bins=5)[0]
    assert abs(five.psi - population_stability_index(dev, moved, quantile_edges(dev, 5)).psi) < 1e-12 and five.label == "7", (
        f"n_bins = 5 means five bins cut from the development scores, and a label comes back as a string; got PSI "
        f"{five.psi:.4f} and label {five.label!r}")
    for bad in ([], [("a", dev, y[:10])], [("a", dev[:0], y[:0])]):
        try:
            stability_table(dev, bad)
        except ValueError:
            continue
        raise AssertionError("stability_table must raise ValueError for no vintages, unequal lengths or an empty vintage")
    print("exercise 8 looks right — edges cut once on development, a nan Gini instead of a crash")

In [ ]:
_try("exercise 8", _check_stability_table)

In [ ]:
def _show_stability() -> None:
    if not _ready("stability by vintage", "entries"):
        return
    entries = S["entries"]
    shown = ("scorecard", "unconstrained boosting", "monotone boosting")
    tables = {}
    for name in shown:
        dev_score, vints = _month_vintages(entries[name]["predict"], BOOK)
        tables[name] = stability_table(dev_score, vints)
    months = [f"month {m}" for m in range(*PERIODS["out-of-time test"])]
    util_psi = []
    for label, mk in ((f"month {m}", BOOK.vintage == m) for m in range(*PERIODS["out-of-time test"])):
        util_psi.append(population_stability_index(
            BOOK.X[DEV, UTIL], BOOK.X[mk, UTIL], quantile_edges(BOOK.X[DEV, UTIL], PSI_BINS)).psi)
    head = f"{'vintage':<10}{'default':>9}{'util PSI':>10}"
    for name in shown:
        head += f"{name.split()[0] + ' Gini':>20}{'PSI':>8}"
    print(head)
    for i, label in enumerate(months):
        row = f"{label:<10}{tables[shown[0]][i].default_rate:>9.1%}{util_psi[i]:>10.3f}"
        for name in shown:
            r = tables[name][i]
            row += f"{r.gini:>20.3f}{r.psi:>8.3f}"
        print(row)
    S["stability"] = tables
    print("\nPSI is computed on each model's own score, against that model's development scores.")
    for name in shown:
        t = tables[name]
        print(f"{name:<24} Gini {t[0].gini:.3f} in the first test month, {t[-1].gini:.3f} in the last "
              f"(mean {np.mean([r.gini for r in t]):.3f}); PSI {t[0].psi:.3f} then {t[-1].psi:.3f}.")
    last = {n: tables[n][-1].psi for n in shown}
    hi, lo = max(last, key=last.get), min(last, key=last.get)
    rose = [n for n in shown if tables[n][-1].psi > tables[n][0].psi]
    print(f"in the last month PSI runs from {last[lo]:.3f} ({lo}) to {last[hi]:.3f} ({hi}), a spread of "
          f"{last[hi] - last[lo]:.3f} between the models; it was higher in the last month than in the first for "
          f"{len(rose)} of {len(shown)} models.")
    dev_score, vints = _month_vintages(entries["monotone boosting"]["predict"], BOOK)
    own = [population_stability_index(dev_score, score, quantile_edges(score, PSI_BINS)).psi for _, score, _ in vints]
    frozen = [r.psi for r in tables["monotone boosting"]]
    for k in (0, -1):
        print(f"monotone boosting, {vints[k][0]}: PSI {frozen[k]:.3f} on edges frozen on development, "
              f"{own[k]:.3f} on edges re-cut from that month's own scores")
    gap = max(abs(a - b) for a, b in zip(frozen, own))
    print(f"over the {len(vints)} test months the two figures differ by at most {gap:.3f} for this model on this book. "
          "Whatever that gap is, the second figure is not the PSI of this report: its bins are a different "
          "yardstick every month, and the reason to freeze the edges is the definition, not the size of the gap.")
    print("PSI says how far a score distribution moved from development; whether the ranking held is what the "
          "Gini columns say, and the two need not move together.")


_try("stability by vintage", _show_stability, needs=tuple(f"exercise {i}" for i in range(1, 9)),
     limit=150, heavy=True)

## 17. One book is one draw

Everything so far came from a single book, and a single book can flatter either model. The last
measurement repeats the whole comparison on `REPLICATES` books from the same generator and fixed
seeds, so you can see which numbers move from book to book and which do not. For each book it
fits the four models on that book's development months, chooses the trees on its validation
months, and measures, on its own out-of-time months:

- the **Gini** on the development rows and on the out-of-time rows, and the mean **PSI** of the
  score over the out-of-time months;
- how many of the four inputs **fail the test** of section 14 (reference rows from that book);
- the **curve spread**: each model's utilisation curve (as in section 13) is evaluated on the
  *same* rows and grid, whichever book the model came from, and the standard deviation of the
  curve across books is averaged over the grid. A model whose shape depends on which book it was
  fitted on has a large spread.

In [ ]:
def _failing_inputs(entry: dict, X_ref: np.ndarray) -> tuple[int, int]:
    """(inputs that fail the test, reference rows that fall in utilisation) for one model. Given."""
    n_fail, util_rows = 0, 0
    for j in range(len(FEATURES)):
        v = monotone_violations(entry["predict"], X_ref, j, "increasing" if DIRECTIONS[j] > 0 else "decreasing",
                                exact_grid(entry["thresholds"](j)))
        n_fail += int(v.n_rows > 0)
        if j == UTIL:
            util_rows = v.n_rows
    return n_fail, util_rows


def _measure_book(book: Book, curve_rows: np.ndarray, curve_grid: np.ndarray) -> dict:
    entries, info = fit_four(book)
    masks = tuple(period_mask(book.vintage, n) for n in PERIODS)
    test = masks[2]
    X_ref = book.X[np.sort(np.random.default_rng(5).choice(np.flatnonzero(test), ICE_ROWS, replace=False))]
    out = {"planted truth": {"dev": auc_gini(book.pd_true[masks[0]], book.y[masks[0]])[1],
                             "test": auc_gini(book.pd_true[test], book.y[test])[1]}}
    for name, e in entries.items():
        dev_score, vints = _month_vintages(e["predict"], book)
        table = stability_table(dev_score, vints)
        n_fail, util_rows = _failing_inputs(e, X_ref)
        out[name] = {"dev": _gini_on(e["predict"], book, masks[0]), "test": _gini_on(e["predict"], book, test),
                     "psi": float(np.mean([r.psi for r in table])), "fail": n_fail, "util_rows": util_rows,
                     "curve": _average_curve(e["predict"], curve_rows, UTIL, curve_grid),
                     "trees": info[name]["trees"], "leaves": info[name]["leaves"]}
        if name == "scorecard":
            out[name]["wrong_sign"] = int(np.sum(info[name]["coef"][1:] > 0.0))
    return out


def _span(values: Sequence[float], fmt: str = ".3f") -> str:
    v = np.asarray(values, dtype=float)
    return f"{v.mean():{fmt}} ({v.min():{fmt}} to {v.max():{fmt}})"


def _sweep_books() -> None:
    curve_rows = BOOK.X[TEST][np.random.default_rng(3).choice(int(TEST.sum()), 1500, replace=False)]
    curve_grid = np.quantile(BOOK.X[TEST, UTIL], np.linspace(0.02, 0.98, 13))
    results = [_measure_book(BOOK if k == 0 else application_book(_book_seed(k)), curve_rows, curve_grid)
               for k in range(REPLICATES)]
    S["sweep"] = results
    names = ("scorecard", "unconstrained boosting", "monotone boosting", "split-only constraint")
    print(f"{REPLICATES} books; each cell is the mean over books, with the lowest and highest in brackets\n")
    print(f"{'model':<24}{'development Gini':>26}{'out-of-time Gini':>26}{'mean PSI':>24}")
    print(f"{'planted truth':<24}{_span([r['planted truth']['dev'] for r in results]):>26}"
          f"{_span([r['planted truth']['test'] for r in results]):>26}")
    for n in names:
        print(f"{n:<24}{_span([r[n]['dev'] for r in results]):>26}{_span([r[n]['test'] for r in results]):>26}"
              f"{_span([r[n]['psi'] for r in results]):>24}")
    print(f"\n{'model':<24}{'inputs failing the test':>26}{'utilisation rows that fall':>28}{'curve spread':>16}")
    spread = {}
    for n in names:
        curves = np.array([r[n]["curve"] for r in results])
        spread[n] = float(curves.std(axis=0).mean())
        print(f"{n:<24}{_span([r[n]['fail'] for r in results], '.1f'):>26}"
              f"{_span([r[n]['util_rows'] for r in results], '.0f'):>28}{spread[n]:>16.4f}")
    S["spread"] = spread
    chosen = [r[n]["trees"] for r in results for n in names[1:]]
    print(f"\ntrees chosen on the validation months, over every book and booster: {min(chosen)} to {max(chosen)}"
          + (f"; at least one fit reached the first cap of {MAX_ROUNDS} and was refitted with more rounds."
             if max(chosen) >= MAX_ROUNDS else f" (the first cap of {MAX_ROUNDS} was never reached)."))
    if max(chosen) >= HARD_ROUNDS:
        print(f"a fit reached the hard cap of {HARD_ROUNDS} trees: its validation loss was still falling.")
    free, mono, split, card = (np.array([[r[n]["dev"], r[n]["test"]] for r in results]) for n in names[1:] + names[:1])
    print(f"\nin how many of the {REPLICATES} books (a gap of any size counts, however small):")
    print(f"  the constraint lowered the development Gini      {int(np.sum(mono[:, 0] < free[:, 0]))}")
    print(f"  monotone boosting beat unconstrained out of time {int(np.sum(mono[:, 1] > free[:, 1]))}")
    print(f"  monotone boosting beat the scorecard out of time {int(np.sum(mono[:, 1] > card[:, 1]))}")
    print(f"  the unconstrained model failed the test          {int(np.sum([r['unconstrained boosting']['fail'] > 0 for r in results]))}")
    print(f"  the split-only constraint failed the test        {int(np.sum([r['split-only constraint']['fail'] > 0 for r in results]))}")
    print(f"  the monotone model failed the test               {int(np.sum([r['monotone boosting']['fail'] > 0 for r in results]))}")
    print(f"  the scorecard failed the test                    {int(np.sum([r['scorecard']['fail'] > 0 for r in results]))}")
    print(f"  the scorecard had a coefficient of the wrong sign {int(np.sum([r['scorecard']['wrong_sign'] > 0 for r in results]))}")
    print(f"  the unconstrained model's development Gini was above the planted truth's "
          f"{int(np.sum([r['unconstrained boosting']['dev'] > r['planted truth']['dev'] for r in results]))}")
    gap = [r["monotone boosting"]["test"] - r["unconstrained boosting"]["test"] for r in results]
    print(f"\nout-of-time Gini, monotone minus unconstrained, book by book: "
          + ", ".join(f"{v:+.4f}" for v in gap))


_try("replicate books", _sweep_books, needs=tuple(f"exercise {i}" for i in range(1, 9)), limit=300, heavy=True)

Read the tables against what each column is able to see. The development Gini rewards a model
for fitting the sample in front of it, noise included, so a constraint tends to lower it (the
sweep counts how often). The out-of-time Gini is the figure to judge by, and the book-by-book
differences say whether any gap between the models is larger than the draw-to-draw movement. The
test counts how often a model's shape contradicts what the lender already knows to be true. The
curve spread says whether that shape is a property of the world or of the sample. None of the
numbers above was tuned: the generator and its seed were fixed before any model was fitted, and
where the lines printed above do not say what you expected, the books are the evidence, not the
expectation.

## 18. Common mistakes

- **Checking the constraint at the split and nowhere else.** Each split looks fine and the tree
  still falls. Hand the bounds down, or the model is only monotone where its trees are shallow.
- **Comparing weights before clipping them.** The child that crosses the other only because a
  bound moved it is wrongly refused; the one that stops crossing only because of the clip is
  wrongly accepted. Clip first, then compare, then score.
- **Reading monotonicity off a partial dependence plot.** A curve averaged over applications can
  rise while individual applications fall, and the reverse. Test every reference row.
- **Testing on a grid you chose rather than one the model chose.** A step function can dip
  between two of your grid points. Put a point at each of the model's own thresholds.
- **Choosing the number of trees on the data the trees were fitted on.** The loss on the fitting
  data tends to keep falling with every tree, so it cannot say when to stop. Choose on months the
  trees did not see; this lesson spends the validation months on that, and never fits on them.
- **Comparing development Gini.** A flexible model usually wins it here (the sweep counts how
  often), partly by fitting noise. The out-of-time months, and the planted truth when a
  simulation offers it, are the yardstick.
- **Cutting PSI bins on each vintage.** The baseline is then different every month, so none of the
  figures is the PSI against development, whatever the size of the gap (section 16 prints how far
  apart the two sets of figures are on this book). Cut once, on the development scores, and keep
  the edges.
- **Treating PSI as a performance measure.** It says the population moved. It does not say whether
  the ranking held; that is the Gini of the new months.

Run the cell below to watch the first of those happen, with numbers computed on the spot, and to
see how many reference rows each wrong way of testing finds on the same unconstrained model.

In [ ]:
def _show_blind_spots() -> None:
    if not _ready("blind spots", "entries"):
        return
    X_ref = _reference_rows()
    e = S["entries"]["unconstrained boosting"]
    lo, hi = np.quantile(BOOK.X[TEST, UTIL], (0.02, 0.98))
    own = exact_grid(e["thresholds"](UTIL))
    grids = {"5 equally spaced points": np.linspace(lo, hi, 5), "20 equally spaced points": np.linspace(lo, hi, 20),
             "50 equally spaced points": np.linspace(lo, hi, 50),
             f"the model's own thresholds ({len(own)} points)": own}
    print(f"unconstrained boosting, utilisation, {len(X_ref)} reference rows: how many fall somewhere along the grid?")
    for label, grid in grids.items():
        v = monotone_violations(e["predict"], X_ref, UTIL, "increasing", grid)
        print(f"  {label:<44}{v.n_rows:>5} rows   worst fall {v.worst_drop:.4f}")
    avg = _average_curve(e["predict"], X_ref, UTIL, own)
    print(f"  the average over rows, on the model's own thresholds: {int(np.sum(np.diff(avg) < 0))} falling steps "
          f"of {len(own) - 1}")
    split = S["entries"]["split-only constraint"]
    free_pass = monotone_violations(S["entries"]["monotone boosting"]["predict"], X_ref, UTIL, "increasing",
                                    exact_grid(S["entries"]["monotone boosting"]["thresholds"](UTIL)))
    v = monotone_violations(split["predict"], X_ref, UTIL, "increasing", exact_grid(split["thresholds"](UTIL)))
    print(f"the split-only constraint, same test: {v.n_rows} rows fall in utilisation; with bounds handed down: "
          f"{free_pass.n_rows}.")


_try("blind spots", _show_blind_spots, needs=tuple(f"exercise {i}" for i in range(1, 8)), limit=150, heavy=True)

## 19. Self-check

1. A colleague's boosted model rejects every split whose left child has a higher weight than its
   right child on an input that must not push risk down, and the model still has applications
   whose risk falls as that input rises. The most likely reason is:
   - (a) the learning rate is too high, so later trees overshoot
   - (b) a deeper split on another input put a leaf on the low side above leaves on the high side,
         because the bounds from the first split were not handed down to it
   - (c) the hessian was floored, so some leaf weights are too large

2. A partial dependence curve for utilisation, averaged over a thousand applications, rises at
   every step of its grid. The right reading is:
   - (a) the model is monotone in utilisation
   - (b) the model is monotone on the average and therefore on every subgroup
   - (c) the model's average behaviour is monotone, and individual applications may still fall,
         so every reference application has to be checked on its own

3. A model's score is a step function of an input, changing only at its thresholds. A monotonicity
   test uses twenty equally spaced grid points across the input's range and finds nothing. What
   does that show?
   - (a) only that those twenty points show no fall: a step function can dip between them, so a
         point at each of the model's own thresholds would leave no step unexamined
   - (b) the model is monotone in that input
   - (c) the model is monotone as long as the twenty points are strictly increasing

4. The unconstrained booster has the higher Gini on the development months and the monotone one
   has the higher Gini on the out-of-time months. Which is the better model?
   - (a) the unconstrained one: development Gini is measured on more data
   - (b) neither can be called better from one book; the out-of-time months are the fairer test, and
         the replicate books say whether a gap of that size is bigger than the draw-to-draw movement
   - (c) the monotone one, because a lower development Gini always means less overfitting

5. The PSI of a model's score has risen steadily for six months while its Gini on each new month has
   not moved. A fair summary is:
   - (a) the model is failing, because PSI is above zero
   - (b) the Gini is wrong, since PSI and Gini must move together
   - (c) the population has moved but the ranking has held, so nothing here says the model needs
         rebuilding; this is why the report carries both

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

One last cell, and it is the deliverable. A challenger does not hand on as a Gini alone: it hands on
with the test it passed, the books it was measured on and the assumptions that were never checked, or
someone will later quote the Gini as though it described a real lender's applicants.

In [ ]:
def _challenger_note() -> None:
    if not _ready("challenger note", "gini", "truth_gini", "info", "verdicts", "sweep"):
        return
    gini, truth = S["gini"], S["truth_gini"]
    verdicts, info, runs = S["verdicts"], S["info"], S["sweep"]
    mono, free, card = "monotone boosting", "unconstrained boosting", "scorecard"
    print("CHALLENGER NOTE — hand this on with the model")
    print(f"  champion     weight-of-evidence scorecard (P04-L07), classes pooled to each input's documented direction")
    print(f"  challenger   boosted trees, depth {MAX_DEPTH}, {info[mono]['trees']} trees chosen on months "
          f"{PERIODS['validation'][0]}-{PERIODS['validation'][1] - 1}, bounds handed down after every constrained split")
    print(f"  fitted on    months {PERIODS['development'][0]}-{PERIODS['development'][1] - 1} of a synthetic book of "
          f"{len(BOOK.y):,} applications; judged on months {PERIODS['out-of-time test'][0]}-"
          f"{PERIODS['out-of-time test'][1] - 1}, which neither model was fitted or tuned on")
    print(f"  out of time  Gini {gini[card][2]:.3f} scorecard, {gini[mono][2]:.3f} monotone boosting, {gini[free][2]:.3f} "
          f"unconstrained boosting; the planted probability itself scores {truth[2]:.3f}")
    print(f"  the test     on {ICE_ROWS} out-of-time reference rows, each varied along the model's own thresholds, "
          "predicted risk must not move against a documented direction: "
          + ", ".join(f"{n} {v[0]}" for n, v in verdicts.items()))
    print(f"  over books   {len(runs)} books from the same generator; the test failed for the unconstrained model in "
          f"{sum(r[free]['fail'] > 0 for r in runs)}, the split-only constraint in "
          f"{sum(r['split-only constraint']['fail'] > 0 for r in runs)}, the monotone model in "
          f"{sum(r[mono]['fail'] > 0 for r in runs)} and the scorecard in {sum(r[card]['fail'] > 0 for r in runs)}")
    print(f"  known gaps   the test covers {ICE_ROWS} out-of-time reference rows, exhaustive along the input it varies "
          "but sampled over the others, so a PASS is not a proof for every applicant; one generator and one family "
          "of shifts; the monotone directions were written down, not learned; the midpoint rule holds a leaf at "
          "the midpoint, so it over-constrains; the planted truth has no concept drift, only a moving "
          "population, so none of this says how any model fares when the relationship itself changes")


_try("challenger note", _challenger_note,
     needs=tuple(f"exercise {i}" for i in range(1, 9)), limit=30, heavy=True)

## What you built, and where it goes next

You now have a boosted-tree learner whose monotonicity is a property you can test rather than a hope:
a split search that refuses out-of-order splits, bounds that carry the constraint down the whole
tree, a loop that picks its size on months it was not fitted on, a test that looks at every reference
application on a grid made of the model's own thresholds, and a stability table that reports Gini and
PSI side by side with its edges frozen on development. The programme's next modules inherit all of
it: the reasons you owe a declined applicant (module 4) come from a model whose direction on every
input you can now vouch for, and the monitoring of a live portfolio (module 7) is this stability
table run on the months that have not happened yet.

A cross-check you can add yourself: LightGBM and XGBoost both take a list of directions, one per input,
in the same `+1`, `-1`, `0` form this lesson uses (`monotone_constraints`), and LightGBM names its
default bounding method `basic` (`claims.yaml` records the documentation as read on 14 October 2026).
This notebook does not run either library, since neither is part of its environment and nothing here
may depend on a download; fitting one on the same development months and passing its predictions to
`stability_table` and `assert_risk_monotone` would put it on the same footing as the models above.

The habit to carry forward: **a model's shape is a claim, and a claim that can be tested should be.**
As of 14 October 2026 the attachment to the US model risk guidance SR 26-2 lists, among the forms
model testing may take, "a comparison of alternative assumptions and methodologies": a challenger
fitted on the same months, judged on the same later months, and carrying its test with it is one way
of making that comparison something a validator can re-run.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check. (A check that ran out
# of time keeps its result unless the function it tests is a different one by now.)
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (("exercise 1", _check_grad_hess, ()),
                                      ("exercise 2", _check_leaf_value, ()),
                                      ("exercise 3", _check_best_split, ()),
                                      ("exercise 4", _check_grow_tree, ("exercise 2", "exercise 3")),
                                      ("exercise 5", _check_fit_booster, ("exercise 1", "exercise 4")),
                                      ("exercise 6", _check_staged_log_loss, ()),
                                      ("exercise 7", _check_monotone_violations, ()),
                                      ("exercise 8", _check_stability_table, ())):
            if _TIMED_OUT.get(_name) == _functions_under_test(_name, _needs):
                continue      # it ran out of time a moment ago and the code is the same: do not spend the time again
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.